# OriginSib-Comp — Disentangling Parental Haplotype Interactions and Compensatory Regulatory Effects in Human Genomes

**One-file Kaggle pipeline.** It runs the analysis plan end to end on public data that it downloads itself:

| Stage | What runs | Blueprint item |
|---|---|---|
| 0 | Config, environment, robust multi-mirror downloader | §13–14 |
| 1 | **E1 data audit**: 1000G 30× pedigree, trios, complex families, overlap N1/N2/N3 with GEUVADIS / MAGE | §4–5, Table 1, S1 |
| 2 | GRCh38 gene annotation (GENCODE v38), ENCODE cCREs (regulatory context) | §4E |
| 3 | Phased 1000G 30× VCF → cis-window biallelic SNVs (fast streaming parser, no bcftools needed) | §13 Step B |
| 4 | **Module 1**: Mendelian checks, paternal/maternal haplotype assignment, switch-error estimates | §6 Module 1 |
| 5 | Leakage-safe preprocessing (INT, genotype PCs, expression PCs, covariates are fit on training folds only) | §9 |
| 6 | **Baselines**: Ridge, PrediXcan-style ElasticNet, aFC-n-style haplotype model, pairwise interaction regressions (phase-blind / cis / trans), XGBoost | §8, Table 3 |
| 7 | **OriginSib-CompNet** = (1) cross-fitted additive ElasticNet → (2) sparse phase-aware cis (γ) / trans (θ) interaction head → (3) gated neural DeepSets residual with symmetric trans branch, POE head, heteroscedastic variance, 5-seed deep ensemble; **ablations A0–A8** | §6–7, Table 5 |
| 8 | **Split A** held-out families, **Split B** held-out genes/genomic blocks, **Split C** cross-ancestry (leave one population out) | §9, Table 6 |
| 9 | **E5 interaction/compensation scan**: discovery → replication, HC3 OLS, BH-FDR, cis-vs-trans phase test, permutation, random-pair and phase-shuffle negative controls, power analysis | §9, §15–16, Table 7 |
| 10 | XAI: model-implied counterfactual interaction contrasts, compared with statistical estimates | §6 Module 5 |
| 11 | E6 parent-aware vs parent-blind, exploratory POE, E9 functional enrichment, optional MPRA, E10 calibration | §9 |
| 12 | Tables 1–7, S1–S3, Figures 1–10 + supplementary figures, reproducibility manifest, zipped results | §11–12 |

### How to run on Kaggle
1. **Settings → Internet → ON** (needed to download the public data). A GPU (T4/P100) is optional but faster.
2. **Run All**. Everything goes to `/kaggle/working/originsib_results/` and is zipped to `/kaggle/working/originsib_results.zip`.
3. Downloads (first run): pedigree (<1 MB), GEUVADIS expression (~30 MB), GENCODE v38 (~30 MB), ENCODE cCREs (~100 MB, optional), 1000G 30× phased chr22 VCF (several hundred MB).
4. If a download is blocked, attach the same file **with the same file name** as a Kaggle Dataset. The notebook searches `/kaggle/input/**` before it tries the internet.

### Scientific guardrails built into the code
* Cis interaction, trans-haplotype interaction and true parent-of-origin (POE) effects are **modelled and tested separately**. POE is only estimated where the parental origin was verified by Mendelian transmission.
* Every discovery carries an **evidence level** (L1 model-only → L4 replicated with regulatory support). Level 5 (direct perturbation) cannot be reached from this data, and the notebook never claims it.
* Nothing is hard-coded or simulated in the real run. If a resource cannot be obtained, that experiment is reported as **not run**, with the reason.
* `SELF_TEST=True` runs the whole pipeline on synthetic data so you can check the code offline. **Never report self-test numbers.**

In [ ]:
# =====================================================================
# 0. CONFIGURATION  (edit here only)
# =====================================================================
import os, sys, re, gc, io, json, math, time, gzip, glob, random, shutil, hashlib, tarfile, zipfile
import platform, warnings, itertools, subprocess, bisect, inspect, traceback
import urllib.request, urllib.error
from pathlib import Path
from collections import defaultdict, OrderedDict
warnings.filterwarnings("ignore")
os.environ["PYTHONWARNINGS"] = "ignore"          # also silences sklearn ConvergenceWarnings inside joblib workers

CFG = dict(
    SELF_TEST            = False,        # True -> synthetic smoke test (offline). NEVER report these numbers.
    CHROMS               = ["chr22"],    # pilot = chr22 (blueprint §14). Add e.g. "chr21","chr20" for more genes.
    CIS_WINDOW           = 100_000,      # +/- bp around the GRCh38 TSS
    MAF_MIN              = 0.05,         # computed in the expression cohort
    K_VARIANTS           = 32,           # candidate variants per gene (blueprint: 32-128)
    K_PAIR               = 8,            # top variants used for explicit pairwise terms / scan (28 pairs)
    CLUMP_R2             = 0.80,         # LD clumping for candidate variants
    PAIR_MAX_R2          = 0.50,         # pairs in stronger LD are not tested (LD artefact control)
    EN_MAX_VARIANTS      = 500,          # PrediXcan-style EN: max cis variants (pre-ranked on TRAIN only)
    MIN_VARIANTS_PER_GENE= 5,
    MAX_GENES            = None,         # None = all QC-passing genes on CHROMS
    N_FOLDS              = 5,
    SEEDS_FULL           = [42, 123, 3407, 2026, 777],   # deep ensemble of the full model
    SEEDS_ABLATION       = [42, 123],                    # seeds per ablation
    SEEDS_SPLIT_BC       = [42, 123, 3407],              # seeds for gene-held-out / cross-ancestry
    HIDDEN               = 128,
    EMB                  = 32,
    DROPOUT              = 0.2,
    BATCH                = 4096,
    LR_GRID              = [3e-3, 1e-3],
    WEIGHT_DECAY         = 1e-4,
    INTERACTION_L1       = 1e-3,         # sparsity on interaction branch outputs
    GENE_L2              = 1e-3,         # L2 on gene-specific embeddings (per row)
    GATE_T               = 2.0,          # per-gene stacking gate: keep CompNet correction only if val t-stat > GATE_T
    MAX_EPOCHS           = 40,
    PATIENCE             = 6,
    MIN_DELTA            = 1e-3,         # early stopping: minimum validation-loss improvement
    VAL_FRAC             = 0.20,         # inner validation families for early stopping
    N_GENO_PCS           = 5,
    N_EXPR_PCS           = 10,           # PEER-like hidden-factor proxy (fit on train only)
    N_GENO_PC_VARIANTS   = 20_000,
    MIN_CELL             = 10,           # min carriers per genotype cell for an interaction test
    MIN_PHASE_CELL       = 5,            # min coupling AND repulsion double-heterozygotes for the cis/trans test
    N_COND_EQTL          = 3,            # conditional eQTL covariates in the scan (LD/tagging control)
    FDR                  = 0.05,
    N_PERM               = 10,
    ORIGIN_QC_MAX_TRIOS  = 250,          # trios used for the phasing / origin QC table
    RUN_SPLIT_B          = True,
    RUN_SPLIT_C          = True,
    USE_MAGE             = True,         # best-effort automatic MAGE (Zenodo 10535719) external validation
    MAGE_MAX_FILE_GB     = 2.0,
    USE_CCRE             = True,
    N_JOBS               = -1,
    BOOTSTRAP            = 2000,
)

IS_KAGGLE = Path("/kaggle/working").exists()
WORK = Path("/kaggle/working") if IS_KAGGLE else Path.cwd()
OUT  = WORK / "originsib_results"
DATA = Path("/tmp/originsib_data")          # large raw files: kept out of the Kaggle output folder
for d in [OUT, OUT/"tables", OUT/"figures", OUT/"candidate_interactions", OUT/"checkpoints", OUT/"processed", DATA]:
    d.mkdir(parents=True, exist_ok=True)

if CFG["SELF_TEST"]:
    CFG.update(USE_MAGE=False, SEEDS_FULL=[42, 123], SEEDS_ABLATION=[42], SEEDS_SPLIT_BC=[42],
               MAX_EPOCHS=30, PATIENCE=6, N_PERM=3, BOOTSTRAP=300, LR_GRID=[3e-3, 1e-3])
T0 = time.time()
LOG = []
def log(*a):
    s = " ".join(str(x) for x in a)
    print(s, flush=True); LOG.append(f"[{time.time()-T0:8.1f}s] {s}")
log("OriginSib-Comp | Kaggle:", IS_KAGGLE, "| SELF_TEST:", CFG["SELF_TEST"], "| OUT:", OUT)

In [ ]:
# =====================================================================
# 0b. ENVIRONMENT
# =====================================================================
def _ensure(pkg, pip_name=None):
    try:
        __import__(pkg); return True
    except ImportError:
        try:
            subprocess.run([sys.executable, "-m", "pip", "install", "-q", pip_name or pkg], check=True, timeout=900)
            __import__(pkg); return True
        except Exception as e:
            print(f"[warn] could not install {pkg}: {e}"); return False

for p, pn in [("numpy", None), ("pandas", None), ("scipy", None), ("sklearn", "scikit-learn"),
              ("statsmodels", None), ("matplotlib", None), ("xgboost", None), ("joblib", None)]:
    _ensure(p, pn)
HAS_TORCH = _ensure("torch")

import numpy as np, pandas as pd
from scipy import stats
from scipy.stats import norm, rankdata, spearmanr, pearsonr, wilcoxon, fisher_exact, chi2, ncx2
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from matplotlib.patches import FancyBboxPatch
from sklearn.linear_model import RidgeCV, ElasticNetCV, LassoCV
from sklearn.model_selection import KFold, GroupKFold
from statsmodels.stats.multitest import multipletests
from joblib import Parallel, delayed
import xgboost as xgb
import torch
from torch import nn

N_JOBS = os.cpu_count() if CFG["N_JOBS"] in (-1, None) else CFG["N_JOBS"]
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
if DEVICE.type == "cpu" and not CFG["SELF_TEST"]:
    print("[warn] No GPU detected. Kaggle: Settings -> Accelerator -> GPU (T4/P100) is strongly recommended.\n"
          "       Falling back to fewer seeds so the run fits in the Kaggle time limit (reported in S3).")
    CFG.update(SEEDS_FULL=[42, 123, 3407], SEEDS_ABLATION=[42], SEEDS_SPLIT_BC=[42], MAX_EPOCHS=25)
pd.set_option("display.width", 200); pd.set_option("display.max_columns", 40)
plt.rcParams.update({"figure.dpi": 110, "savefig.dpi": 300, "font.size": 9, "axes.spines.top": False,
                     "axes.spines.right": False, "axes.titleweight": "bold"})
PAL = ["#2F5D8A", "#C0504D", "#4E9A6B", "#8064A2", "#E09A3E", "#4BACC6", "#7F7F7F", "#B65F9A", "#9BBB59", "#333333"]

def seed_all(s):
    random.seed(s); np.random.seed(s); torch.manual_seed(s)
    if torch.cuda.is_available(): torch.cuda.manual_seed_all(s)
seed_all(42)

VERSIONS = {"python": platform.python_version(), "numpy": np.__version__, "pandas": pd.__version__,
            "torch": torch.__version__, "xgboost": xgb.__version__, "device": str(DEVICE),
            "sklearn": __import__("sklearn").__version__, "statsmodels": __import__("statsmodels").__version__,
            "cpus": os.cpu_count(),
            "gpu": torch.cuda.get_device_name(0) if torch.cuda.is_available() else None}
log("Environment:", VERSIONS)

def savefig(fig, name):
    for ext in ("png", "pdf"):
        fig.savefig(OUT / "figures" / f"{name}.{ext}", bbox_inches="tight")
    plt.show(); plt.close(fig)

def md(df, index=False):
    try: return df.to_markdown(index=index)
    except Exception: return df.to_string(index=index)

def save_table(df, name, index=False):
    df.to_csv(OUT / "tables" / f"{name}.csv", index=index)
    with open(OUT / "tables" / f"{name}.md", "w") as f: f.write(md(df, index))
    return df

In [ ]:
# =====================================================================
# 0c. ROBUST DOWNLOADER (cache -> /kaggle/input -> mirrors, resume, retries)
# =====================================================================
UA = {"User-Agent": "Mozilla/5.0 (research-notebook; python-urllib)"}
MANIFEST = []

def _md5(path, nbytes=None):
    h = hashlib.md5()
    with open(path, "rb") as f:
        while True:
            b = f.read(1 << 22)
            if not b: break
            h.update(b)
    return h.hexdigest()

def find_in_kaggle_input(fname):
    roots = [Path("/kaggle/input")]
    base = fname[:-3] if fname.endswith(".gz") else fname
    for root in roots:
        if not root.exists(): continue
        for dp, dn, fn in os.walk(root):
            for f in fn:
                if f == fname or f == base:
                    return Path(dp) / f
    return None

def _http_get(url, dest, timeout=90):
    tmp = Path(str(dest) + ".part")
    pos = tmp.stat().st_size if tmp.exists() else 0
    req = urllib.request.Request(url, headers=UA)
    if pos and url.startswith("http"):
        req.add_header("Range", f"bytes={pos}-")
    with urllib.request.urlopen(req, timeout=timeout) as r:
        code = getattr(r, "status", 200)
        if pos and code != 206: pos = 0
        cl = r.headers.get("Content-Length")
        total = (int(cl) + pos) if cl else None
        done, last = pos, time.time()
        with open(tmp, "ab" if pos else "wb") as f:
            while True:
                chunk = r.read(1 << 20)
                if not chunk: break
                f.write(chunk); done += len(chunk)
                if time.time() - last > 20:
                    msg = f"{done/1e6:,.0f} MB" + (f" / {total/1e6:,.0f} MB" if total else "")
                    print("   ...", dest.name, msg, flush=True); last = time.time()
    if total and done < total:
        raise IOError(f"incomplete download {done}/{total}")
    tmp.rename(dest)

def _valid(path, min_bytes):
    if not path.exists() or path.stat().st_size < min_bytes: return False
    if str(path).endswith(".gz"):
        try:
            with gzip.open(path, "rb") as f: f.read(4096)
        except Exception:
            return False
    return True

def fetch(fname, urls, min_bytes=1000, required=True, desc="", resolvers=()):
    """cache -> /kaggle/input -> each URL (resume + retries; a 403/404/410 is not retried) -> resolver functions
       that discover new URLs at run time (e.g. crawling the EBI directory listing)."""
    dest = DATA / fname
    if _valid(dest, min_bytes):
        MANIFEST.append(dict(file=fname, source="cache", path=str(dest), bytes=dest.stat().st_size)); return dest
    k = find_in_kaggle_input(fname)
    if k is not None:
        log(f"[data] {fname}: found in Kaggle input -> {k}")
        MANIFEST.append(dict(file=fname, source="kaggle_input", path=str(k), bytes=k.stat().st_size)); return k
    if CFG["SELF_TEST"]:
        raise FileNotFoundError(f"SELF_TEST: synthetic file missing {fname}")
    errs, tried = [], set()
    def _try(url):
        for attempt in range(3):
            try:
                log(f"[data] downloading {fname} ({desc}) <- {url}  [try {attempt+1}]")
                _http_get(url, dest)
                if _valid(dest, min_bytes):
                    MANIFEST.append(dict(file=fname, source=url, path=str(dest), bytes=dest.stat().st_size,
                                         md5=_md5(dest)))
                    log(f"[data] OK {fname}: {dest.stat().st_size/1e6:,.1f} MB")
                    return True
                raise IOError("downloaded file failed validation")
            except Exception as e:
                errs.append(f"{url}: {type(e).__name__}: {e}")
                if dest.exists() and not _valid(dest, min_bytes): dest.unlink()
                if isinstance(e, urllib.error.HTTPError) and e.code in (400, 401, 403, 404, 410):
                    return False          # permanent: do not retry this URL
                time.sleep(3 * (attempt + 1))
        return False
    def _run(lst):
        for url in lst:
            if url and url not in tried:
                tried.add(url)
                if _try(url): return True
        return False
    if _run(urls): return dest
    for res in resolvers:
        try:
            found = res()
        except Exception as e:
            errs.append(f"resolver {getattr(res, '__name__', res)}: {type(e).__name__}: {e}"); found = []
        if found: log(f"[data] resolver found candidate URLs for {fname}: {found[:4]}")
        if _run(found): return dest
    msg = (f"Could not obtain {fname}. Kaggle: turn Settings->Internet ON, or attach the file "
           f"'{fname}' as a Kaggle Dataset. Errors:\n  " + "\n  ".join(errs[-8:]))
    if required: raise RuntimeError(msg)
    log("[data][optional-missing]", msg.splitlines()[0]); return None

def crawl_listing(bases, exact, pattern=None, max_depth=3, max_pages=80):
    """Breadth-first search of HTTP directory listings (EBI FTP over HTTPS) for a file.
       Returns URLs: exact-name matches first, then files matching `pattern`."""
    hits_exact, hits_pat, seen, pages = [], [], set(), 0
    queue = [(b if b.endswith("/") else b + "/", 0) for b in bases]
    while queue and pages < max_pages:
        url, d = queue.pop(0)
        if url in seen: continue
        seen.add(url); pages += 1
        try:
            req = urllib.request.Request(url, headers=UA)
            with urllib.request.urlopen(req, timeout=60) as r:
                html = r.read().decode("utf-8", "ignore")
        except Exception:
            continue
        for href in re.findall(r'href="([^"?#]+)"', html):
            if href.startswith(("/", "http", "..", "mailto")) or href in ("./",): continue
            full = url + href
            name = href.rstrip("/").split("/")[-1]
            if href.endswith("/"):
                if d + 1 <= max_depth: queue.append((full, d + 1))
            elif name == exact:
                hits_exact.append(full)
            elif pattern and re.search(pattern, name):
                hits_pat.append(full)
        if hits_exact: break
    return hits_exact + hits_pat

def http_json(url, timeout=60):
    req = urllib.request.Request(url, headers=dict(UA, Accept="application/json"))
    with urllib.request.urlopen(req, timeout=timeout) as r:
        return json.loads(r.read().decode())

def http_post_json(url, payload, timeout=120):
    req = urllib.request.Request(url, data=json.dumps(payload).encode(),
                                 headers=dict(UA, **{"Content-Type": "application/json", "Accept": "application/json"}))
    with urllib.request.urlopen(req, timeout=timeout) as r:
        return json.loads(r.read().decode())

G1K = ["https://ftp.1000genomes.ebi.ac.uk/vol1/ftp", "http://ftp.1000genomes.ebi.ac.uk/vol1/ftp",
       "https://ftp-trace.ncbi.nlm.nih.gov/1000genomes/ftp", "https://ftp-trace.ncbi.nih.gov/1000genomes/ftp",
       "ftp://ftp.1000genomes.ebi.ac.uk/vol1/ftp"]
HC = "data_collections/1000G_2504_high_coverage"
PED_NAME = "20130606_g1k_3202_samples_ped_population.txt"
PED_URLS = [f"{b}/{HC}/{PED_NAME}" for b in G1K]
def vcf_name(ch): return f"1kGP_high_coverage_Illumina.{ch}.filtered.SNV_INDEL_SV_phased_panel.vcf.gz"
def vcf_urls(ch): return [f"{b}/{HC}/working/20220422_3202_phased_SNV_INDEL_SV/{vcf_name(ch)}" for b in G1K]
GEU_NAME = "GD462.GeneQuantRPKM.50FN.samplename.resk10.txt.gz"
# GEUVADIS processed files live in the "analysis_results" sub-folder of E-GEUV-1 (ArrayExpress -> BioStudies)
GEU_BASES = ["https://ftp.ebi.ac.uk/biostudies/fire/E-GEUV-/001/E-GEUV-1/Files",
             "https://ftp.ebi.ac.uk/biostudies/nfs/E-GEUV-/001/E-GEUV-1/Files",
             "https://ftp.ebi.ac.uk/biostudies/fire/E-GEUV-/E-GEUV-1/Files"]
GEU_URLS = ([f"{b}/analysis_results/{GEU_NAME}" for b in GEU_BASES] +
            [f"https://www.ebi.ac.uk/biostudies/files/E-GEUV-1/analysis_results/{GEU_NAME}",
             f"https://www.ebi.ac.uk/arrayexpress/files/E-GEUV-1/analysis_results/{GEU_NAME}",
             f"https://ftp.ebi.ac.uk/pub/databases/arrayexpress/data/experiment/GEUV/E-GEUV-1/analysis_results/{GEU_NAME}"] +
            [f"{b}/{GEU_NAME}" for b in GEU_BASES[:2]])
GEU_PATTERN = r"^GD46\d\.GeneQuantRPKM.*\.txt\.gz$"     # fallback: any GEUVADIS gene-level RPKM matrix
GTF_NAME = "gencode.v38.basic.annotation.gtf.gz"
GTF_URLS = [f"{b}/pub/databases/gencode/Gencode_human/release_38/{GTF_NAME}"
            for b in ["https://ftp.ebi.ac.uk", "http://ftp.ebi.ac.uk", "ftp://ftp.ebi.ac.uk"]]
CCRE_NAME = "GRCh38-cCREs.bed"
CCRE_URLS = ["https://downloads.wenglab.org/V3/GRCh38-cCREs.bed",
             "https://downloads.wenglab.org/cCREs/GRCh38-cCREs.bed"]

def biostudies_bases(acc):
    """Ask the BioStudies API where the study's files live (handles EBI path changes)."""
    out = []
    try:
        info = http_json(f"https://www.ebi.ac.uk/biostudies/api/v1/studies/{acc}/info")
        for k in ("ftpHttp_link", "ftpLink", "httpLink"):
            if info.get(k):
                base = info[k].rstrip("/")
                if base.startswith("ftp://"): base = "https://" + base[len("ftp://"):]
                out.append(base + ("" if base.endswith("/Files") else "/Files"))
    except Exception as e:
        log("[data] BioStudies API not reachable:", type(e).__name__)
    return out

def resolve_geuvadis_urls():
    bases = biostudies_bases("E-GEUV-1") + GEU_BASES
    bases = list(dict.fromkeys(bases))
    direct = [f"{b}/analysis_results/{GEU_NAME}" for b in bases]
    return direct + crawl_listing(bases, GEU_NAME, GEU_PATTERN, max_depth=2)

In [ ]:
# =====================================================================
# 0d. SELF-TEST: synthetic data in the SAME file formats (only if SELF_TEST=True)
# =====================================================================
def make_synthetic_data(seed=7):
    rng = np.random.default_rng(seed)
    pops = ["CEU", "FIN", "GBR", "TSI", "YRI"]
    sup = {"CEU": "EUR", "FIN": "EUR", "GBR": "EUR", "TSI": "EUR", "YRI": "AFR"}
    ped_rows, samples = [], []
    sid = 100
    for p in pops:
        for _ in range(56):
            s = f"NA{sid:05d}"; sid += 1
            ped_rows.append((s, s, "0", "0", rng.integers(1, 3), p, sup[p])); samples.append(s)
    trios = []
    for p in pops:
        for t in range(8):
            fa, mo, ch = f"HG{sid:05d}", f"HG{sid+1:05d}", f"HG{sid+2:05d}"; sid += 3
            fam = f"FAM{sid}"
            ped_rows += [(fam, fa, "0", "0", 1, p, sup[p]), (fam, mo, "0", "0", 2, p, sup[p]),
                         (fam, ch, fa, mo, rng.integers(1, 3), p, sup[p])]
            trios.append((fa, mo, ch, p))
    ped = pd.DataFrame(ped_rows, columns=["FamilyID", "SampleID", "FatherID", "MotherID", "Sex", "Population", "Superpopulation"])
    ped.to_csv(DATA / PED_NAME, sep=" ", index=False)
    all_s = list(ped.SampleID)
    pop_of = dict(zip(ped.SampleID, ped.Population))
    n_genes = 70
    tss = np.linspace(17_500_000, 49_000_000, n_genes).astype(int)
    strands = rng.choice(["+", "-"], n_genes)
    genes = [f"ENSG{(100000+i):011d}" for i in range(n_genes)]
    # variants
    pos = np.unique(np.concatenate([t + rng.integers(-CFG["CIS_WINDOW"], CFG["CIS_WINDOW"], 140) for t in tss]))
    nv = len(pos)
    base_af = rng.uniform(0.03, 0.5, nv)
    hap = {}
    for s in all_s:
        if s in {c for _, _, c, _ in trios}: continue
        af = np.clip(base_af + (0.12 if pop_of[s] == "YRI" else 0) * rng.normal(0, 1, nv) * 0.5, 0.01, 0.99)
        hap[s] = (rng.random((nv, 2)) < af[:, None]).astype(np.int8)
    for fa, mo, ch, p in trios:
        a = hap[fa][:, rng.integers(0, 2)]; b = hap[mo][:, rng.integers(0, 2)]
        hap[ch] = np.stack([a, b], 1)       # paternal|maternal
    with gzip.open(DATA / vcf_name("chr22"), "wt") as f:
        f.write("##fileformat=VCFv4.2\n##source=SYNTHETIC_SELF_TEST\n")
        f.write("#CHROM\tPOS\tID\tREF\tALT\tQUAL\tFILTER\tINFO\tFORMAT\t" + "\t".join(all_s) + "\n")
        H = np.stack([hap[s] for s in all_s], 1)  # nv x ns x 2
        for v in range(nv):
            gt = "\t".join(f"{x}|{y}" for x, y in H[v])
            f.write(f"chr22\t{pos[v]}\t.\tA\tG\t.\tPASS\t.\tGT\t{gt}\n")
    # expression with planted additive, cis and trans effects
    expr_s = samples + [c for _, _, c, _ in trios]
    E = np.zeros((n_genes, len(expr_s)))
    truth = []
    for gi, (g, t) in enumerate(zip(genes, tss)):
        w = np.where(np.abs(pos - t) < CFG["CIS_WINDOW"])[0]
        wc = w[(base_af[w] > 0.25)]
        c = rng.choice(wc, 3, replace=False)
        hA = np.array([hap[s][c, 0] for s in expr_s]); hB = np.array([hap[s][c, 1] for s in expr_s])
        d = hA + hB
        y = d @ rng.normal(0, 0.45, 3)
        kind = ["none", "cis", "trans", "dom"][gi % 4]
        gam = 1.4 * rng.choice([-1, 1])
        if kind == "cis":   y += gam * (hA[:, 0] * hA[:, 1] + hB[:, 0] * hB[:, 1])
        if kind == "trans": y += gam * (hA[:, 0] * hB[:, 1] + hB[:, 0] * hA[:, 1])
        if kind == "dom":   y += gam * (hA[:, 0] * hB[:, 0])
        y += np.array([0.3 if pop_of[s] == "YRI" else 0 for s in expr_s]) + rng.normal(0, 1, len(expr_s))
        E[gi] = y
        truth.append((g, kind, pos[c[0]], pos[c[1]]))
    other = rng.normal(0, 1, (300, len(expr_s)))
    lat = rng.normal(0, 1, (3, len(expr_s)))
    other += rng.normal(0, 0.5, (300, 3)) @ lat; E += rng.normal(0, 0.3, (n_genes, 3)) @ lat
    allg = genes + [f"ENSG{(900000+i):011d}" for i in range(300)]
    df = pd.DataFrame(np.vstack([E, other]), columns=expr_s)
    df.insert(0, "Coord", list(tss) + [1000 * i for i in range(300)])
    df.insert(0, "Chr", ["22"] * n_genes + ["1"] * 300)
    df.insert(0, "Gene_Symbol", [f"G{i}" for i in range(len(allg))])
    df.insert(0, "TargetID", [g + ".1" for g in allg])
    df.to_csv(DATA / GEU_NAME, sep="\t", index=False, compression="gzip")
    with gzip.open(DATA / GTF_NAME, "wt") as f:
        for g, t, s in zip(genes, tss, strands):
            st, en = (t, t + 20000) if s == "+" else (t - 20000, t)
            f.write(f'chr22\tSYN\tgene\t{st}\t{en}\t.\t{s}\t.\tgene_id "{g}.1"; gene_type "protein_coding"; gene_name "{g[-4:]}";\n')
        for i in range(300):
            f.write(f'chr1\tSYN\tgene\t{1000*i+1}\t{1000*i+500}\t.\t+\t.\tgene_id "ENSG{(900000+i):011d}.1"; gene_type "lncRNA"; gene_name "X{i}";\n')
    cc = np.sort(rng.choice(pos, 400, replace=False))
    pd.DataFrame({"c": "chr22", "s": cc - 100, "e": cc + 100, "id": [f"EH{i}" for i in range(400)]}).to_csv(
        DATA / CCRE_NAME, sep="\t", header=False, index=False)
    pd.DataFrame(truth, columns=["gene", "kind", "pos1", "pos2"]).to_csv(DATA / "synthetic_truth.csv", index=False)
    log("SELF_TEST synthetic data written to", DATA)

if CFG["SELF_TEST"]:
    make_synthetic_data()

## 1. E1 — Data audit: pedigree, trios, cohort overlap (Table 1, Supplement S1)

In [ ]:
# =====================================================================
# 1. PEDIGREE + EXPRESSION + OVERLAP AUDIT
# =====================================================================
ped_path = fetch(PED_NAME, PED_URLS, min_bytes=10_000 if not CFG["SELF_TEST"] else 100, desc="1000G 30x pedigree")
ped_raw = pd.read_csv(ped_path, sep=r"\s+", dtype=str)
def _col(df, *keys, exclude=()):
    for c in df.columns:
        cl = c.lower()
        if all(k in cl for k in keys) and not any(e in cl for e in exclude): return c
    return None
cmap = {"family": _col(ped_raw, "fam"), "sample": _col(ped_raw, "sample"), "father": _col(ped_raw, "father"),
        "mother": _col(ped_raw, "mother"), "sex": _col(ped_raw, "sex"),
        "pop": _col(ped_raw, "pop", exclude=("super",)), "superpop": _col(ped_raw, "super")}
log("Pedigree columns mapped:", cmap)
assert cmap["sample"] and cmap["father"] and cmap["mother"], "Pedigree schema not recognised"
PED = pd.DataFrame({k: ped_raw[v] if v else np.nan for k, v in cmap.items()})
MISSING = {"0", ".", "NA", "nan", "", "-9"}
for c in ["father", "mother"]:
    PED[c] = PED[c].where(~PED[c].astype(str).isin(MISSING), None)
PED = PED.drop_duplicates("sample").set_index("sample", drop=False)
TRIOS = PED[PED.father.notna() & PED.mother.notna()].copy()
TRIOS["parents_in_ped"] = TRIOS.father.isin(PED.index) & TRIOS.mother.isin(PED.index)
# complex structures: parents that are themselves children, sibships, multi-generation
children_set = set(TRIOS.index)
TRIOS["parent_is_child"] = TRIOS.father.isin(children_set) | TRIOS.mother.isin(children_set)
sib_counts = TRIOS.groupby(["father", "mother"]).size()
TRIOS["n_full_sibs"] = [sib_counts.get((f, m), 1) for f, m in zip(TRIOS.father, TRIOS.mother)]
log(f"Pedigree samples: {len(PED)} | complete trio records: {len(TRIOS)} | "
    f"multi-child sibships: {(sib_counts>1).sum()} | multigenerational links: {int(TRIOS.parent_is_child.sum())}")

# ---- union-find families (pedigree family ID + parent links) ----
_par = {}
def _find(x):
    while _par.setdefault(x, x) != x:
        _par[x] = _par[_par[x]]; x = _par[x]
    return x
def _union(a, b): _par[_find(a)] = _find(b)
for s, r in PED.iterrows():
    _find(s)
    if isinstance(r.family, str): _union(s, "FAM::" + r.family)
    for p in (r.father, r.mother):
        if isinstance(p, str): _union(s, p)
FAMILY = {s: _find(s) for s in PED.index}

# ---- GEUVADIS expression ----
geu_path = fetch(GEU_NAME, GEU_URLS, min_bytes=100_000 if not CFG["SELF_TEST"] else 100, desc="GEUVADIS expression",
                 resolvers=[resolve_geuvadis_urls])
SAMPLE_RE = re.compile(r"^(HG|NA|GM)\d{5}$")
def parse_expression_table(path_or_df):
    df = path_or_df if isinstance(path_or_df, pd.DataFrame) else pd.read_csv(path_or_df, sep=None, engine="python") \
        if str(path_or_df).endswith((".csv", ".csv.gz")) else pd.read_csv(path_or_df, sep="\t")
    scols = [c for c in df.columns if SAMPLE_RE.match(str(c).split(".")[0])]
    if len(scols) < 20:   # maybe samples are rows
        first = df.columns[0]
        if df[first].astype(str).str.match(SAMPLE_RE).mean() > 0.8:
            df = df.set_index(first).T.reset_index().rename(columns={"index": "gene_id"})
            scols = [c for c in df.columns if SAMPLE_RE.match(str(c))]
    gcol = None
    for c in df.columns:
        if c in scols: continue
        if df[c].astype(str).str.contains(r"^ENSG\d+", regex=True).mean() > 0.5: gcol = c; break
    if gcol is None: raise ValueError("no Ensembl gene-id column found")
    X = df[scols].apply(pd.to_numeric, errors="coerce")
    X.columns = [str(c).split(".")[0] for c in scols]
    X = X.loc[:, ~X.columns.duplicated()]
    X.index = df[gcol].astype(str).str.replace(r"\.\d+$", "", regex=True).values
    X = X[~X.index.duplicated()]
    X = X[X.index.str.startswith("ENSG")]
    return X
GEU = parse_expression_table(geu_path)
log(f"GEUVADIS: {GEU.shape[0]} genes x {GEU.shape[1]} samples")

# ---- MAGE (best effort, external independent-person validation) ----
MAGE = None; MAGE_NOTE = "disabled"
def try_load_mage():
    for f in (list(Path("/kaggle/input").rglob("*")) if Path("/kaggle/input").exists() else []):
        if "mage" in str(f).lower() and f.is_file() and re.search(r"(expr|tpm|count|quant|norm)", f.name, re.I):
            try:
                X = parse_expression_table(f)
                if X.shape[1] >= 100: return X, f"kaggle_input:{f}"
            except Exception: pass
    meta = http_json("https://zenodo.org/api/records/10535719")
    files = meta.get("files", [])
    def _key(f): return f.get("key") or f.get("filename") or ""
    def _size(f): return f.get("size") or f.get("filesize") or 0
    def _link(f):
        L = f.get("links", {}); return L.get("self") or L.get("content") or L.get("download")
    log("MAGE Zenodo files:", [(_key(f), round(_size(f) / 1e9, 2)) for f in files][:40])
    cands = [f for f in files if re.search(r"(expr|tpm|count|quant|norm)", _key(f), re.I)
             and _size(f) <= CFG["MAGE_MAX_FILE_GB"] * 1e9]
    cands.sort(key=lambda f: (0 if re.search(r"(norm|tpm)", _key(f), re.I) else 1, _size(f)))
    for f in cands[:4]:
        name = Path(_key(f)).name
        p = fetch(name, [_link(f)], min_bytes=10_000, required=False, desc="MAGE")
        if p is None: continue
        paths = [p]
        if tarfile.is_tarfile(p):
            exdir = DATA / ("mage_" + name.split(".")[0]); exdir.mkdir(exist_ok=True)
            with tarfile.open(p) as tf:
                mem = [m for m in tf.getmembers() if m.isfile() and m.size < 3e9 and
                       re.search(r"(expr|tpm|count|quant|norm)", m.name, re.I) and
                       re.search(r"\.(tsv|txt|csv)(\.gz)?$", m.name)]
                for m in mem[:6]:
                    m.name = Path(m.name).name; tf.extract(m, exdir)
            paths = list(exdir.glob("*"))
        for q in paths:
            try:
                X = parse_expression_table(q)
                if X.shape[1] >= 100 and X.shape[0] >= 1000: return X, f"zenodo:{name}:{q.name}"
            except Exception as e:
                log("  MAGE parse skip", q.name, type(e).__name__)
    raise RuntimeError("no parseable MAGE expression matrix found")
if CFG["USE_MAGE"]:
    try:
        MAGE, MAGE_NOTE = try_load_mage()
        log(f"MAGE loaded: {MAGE.shape} from {MAGE_NOTE}")
    except Exception as e:
        MAGE, MAGE_NOTE = None, f"not available ({type(e).__name__}: {str(e)[:120]})"
        log("MAGE:", MAGE_NOTE)

# ---- overlap audit N1, N2, N3 ----
geu_ids, mage_ids = set(GEU.columns), set(MAGE.columns) if MAGE is not None else set()
trio_children = set(TRIOS.index[TRIOS.parents_in_ped])
trio_members = trio_children | set(TRIOS.father) | set(TRIOS.mother)
N1 = len(trio_children & mage_ids) if MAGE is not None else np.nan
N2 = len(trio_children & geu_ids)
N3 = len(geu_ids & mage_ids) if MAGE is not None else np.nan
overlap = pd.DataFrame([
    dict(set="1000G pedigree samples", n=len(PED)),
    dict(set="Complete trio records (child with both parents)", n=len(TRIOS)),
    dict(set="Trio children with both parents in pedigree", n=len(trio_children)),
    dict(set="Distinct sibships (father,mother)", n=len(sib_counts)),
    dict(set="Multigenerational links (parent is also a trio child)", n=int(TRIOS.parent_is_child.sum())),
    dict(set="GEUVADIS RNA-seq samples", n=len(geu_ids)),
    dict(set="GEUVADIS samples in 1000G pedigree", n=len(geu_ids & set(PED.index))),
    dict(set="N2 = trio children with GEUVADIS expression", n=N2),
    dict(set="GEUVADIS samples who are trio PARENTS", n=len(geu_ids & (set(TRIOS.father) | set(TRIOS.mother)))),
    dict(set="MAGE RNA-seq samples", n=len(mage_ids) if MAGE is not None else np.nan),
    dict(set="N1 = trio children with MAGE expression", n=N1),
    dict(set="N3 = MAGE ∩ GEUVADIS (same person, different assay)", n=N3),
    dict(set="MAGE samples NOT in GEUVADIS (independent persons)", n=len(mage_ids - geu_ids) if MAGE is not None else np.nan),
])
save_table(overlap, "S1_dataset_overlap_audit")
print(overlap.to_string())

## 2. Gene annotation (GRCh38) and regulatory context (ENCODE cCREs)

In [ ]:
gtf_path = fetch(GTF_NAME, GTF_URLS, min_bytes=1_000_000 if not CFG["SELF_TEST"] else 100,
                 required=False, desc="GENCODE v38 basic",
                 resolvers=[lambda: crawl_listing(["https://ftp.ebi.ac.uk/pub/databases/gencode/Gencode_human/release_38"],
                                                  GTF_NAME, r"^gencode\.v38\.(basic\.)?annotation\.gtf\.gz$", max_depth=0)])
def parse_gtf_genes(path, chroms):
    rows = []
    chroms_b = set(chroms) | {c.replace("chr", "") for c in chroms}
    with gzip.open(path, "rt") as f:
        for line in f:
            if line.startswith("#"): continue
            t = line.split("\t", 9)
            if len(t) < 9 or t[2] != "gene" or t[0] not in chroms_b: continue
            a = t[8]
            gid = re.search(r'gene_id "([^"]+)"', a).group(1).split(".")[0]
            gn = re.search(r'gene_name "([^"]+)"', a); gt = re.search(r'gene_type "([^"]+)"', a)
            st, en, sd = int(t[3]), int(t[4]), t[6]
            rows.append(dict(gene_id=gid, gene_name=gn.group(1) if gn else gid, gene_type=gt.group(1) if gt else "",
                             chrom=t[0] if t[0].startswith("chr") else "chr" + t[0], start=st, end=en, strand=sd,
                             tss=st if sd == "+" else en))
    return pd.DataFrame(rows).drop_duplicates("gene_id").set_index("gene_id", drop=False)
def ensembl_rest_tss(gene_ids):
    rows = []
    ids = list(gene_ids)
    for i in range(0, len(ids), 900):
        js = http_post_json("https://rest.ensembl.org/lookup/id", {"ids": ids[i:i + 900]})
        for gid, r in js.items():
            if not r: continue
            sd = "+" if r.get("strand", 1) == 1 else "-"
            rows.append(dict(gene_id=gid, gene_name=r.get("display_name", gid), gene_type=r.get("biotype", ""),
                             chrom="chr" + str(r["seq_region_name"]), start=r["start"], end=r["end"], strand=sd,
                             tss=r["start"] if sd == "+" else r["end"]))
    return pd.DataFrame(rows).set_index("gene_id", drop=False)
if gtf_path is not None:
    ANNOT = parse_gtf_genes(gtf_path, CFG["CHROMS"]); ANNOT_SRC = "GENCODE v38 basic (GRCh38)"
else:
    log("GTF unavailable -> Ensembl REST lookup (GRCh38)")
    ANNOT = ensembl_rest_tss(GEU.index); ANNOT = ANNOT[ANNOT.chrom.isin(CFG["CHROMS"])]
    ANNOT_SRC = "Ensembl REST (GRCh38)"
log(f"Annotation ({ANNOT_SRC}): {len(ANNOT)} genes on {CFG['CHROMS']}")

CCRE = {}
CCRE_OK = False
if CFG["USE_CCRE"]:
    cpath = fetch(CCRE_NAME, CCRE_URLS, min_bytes=1_000_000 if not CFG["SELF_TEST"] else 100, required=False,
                  desc="ENCODE SCREEN cCREs")
    if cpath is not None:
        cc = pd.read_csv(cpath, sep="\t", header=None, usecols=[0, 1, 2], names=["chrom", "start", "end"])
        for ch in CFG["CHROMS"]:
            sub = cc[cc.chrom == ch].sort_values("start")
            CCRE[ch] = (sub.start.values, sub.end.values)
        CCRE_OK = True
        log(f"cCREs loaded: {sum(len(v[0]) for v in CCRE.values())} on {CFG['CHROMS']}")
def in_ccre(chrom, pos):
    if chrom not in CCRE: return np.zeros(len(pos), bool)
    s, e = CCRE[chrom]
    if len(s) == 0: return np.zeros(len(pos), bool)
    i = np.searchsorted(s, pos, side="right") - 1
    ok = i >= 0
    out = np.zeros(len(pos), bool); out[ok] = pos[ok] <= e[i[ok]]
    return out

# genes to analyse: expressed in GEUVADIS + annotated on CHROMS
GENES = ANNOT[ANNOT.index.isin(GEU.index)].sort_values(["chrom", "tss"]).copy()
log(f"Genes expressed in GEUVADIS on {CFG['CHROMS']}: {len(GENES)}")

## 3. Phased 1000G 30× genotypes in cis windows (streaming parser)

In [ ]:
# samples needed: expression cohort(s) + trios for origin QC
vcf_keep = set(GEU.columns) | (set(MAGE.columns) if MAGE is not None else set())
qc_trios = TRIOS[TRIOS.parents_in_ped].copy()
qc_trios = pd.concat([qc_trios[qc_trios.index.isin(geu_ids | mage_ids)],
                      qc_trios[~qc_trios.index.isin(geu_ids | mage_ids)].sample(frac=1, random_state=1)])
qc_trios = qc_trios.iloc[:max(CFG["ORIGIN_QC_MAX_TRIOS"], int(qc_trios.index.isin(geu_ids | mage_ids).sum()))]
vcf_keep |= set(qc_trios.index) | set(qc_trios.father) | set(qc_trios.mother)

def merged_windows(tss, w):
    iv = sorted((max(1, t - w), t + w) for t in tss)
    out = []
    for s, e in iv:
        if out and s <= out[-1][1]: out[-1][1] = max(out[-1][1], e)
        else: out.append([s, e])
    return np.array([x[0] for x in out]), np.array([x[1] for x in out])

def stream_lines(path):
    if shutil.which("gzip"):
        p = subprocess.Popen(["gzip", "-dc", str(path)], stdout=subprocess.PIPE, bufsize=1 << 22)
        try:
            for line in p.stdout: yield line
        finally:
            p.stdout.close(); p.kill()
    else:
        with gzip.open(path, "rb") as f:
            for line in f: yield line

def parse_vcf_windows(path, chrom, starts, ends, keep, maf_samples, maf_min):
    keep = list(keep)
    names, sel, maf_idx = None, None, None
    POS, ID, REF, ALT, H0, H1 = [], [], [], [], [], []
    n_lines = n_win = n_snv = n_slow = 0
    cb = {chrom.encode(), chrom.replace("chr", "").encode()}
    last_end = ends[-1]; t0 = time.time()
    for line in stream_lines(path):
        if line[:2] == b"##": continue
        if line[:1] == b"#":
            names = [x.decode() for x in line.rstrip(b"\r\n").split(b"\t")[9:]]
            nidx = {s: i for i, s in enumerate(names)}
            keep = [s for s in keep if s in nidx]
            sel = np.array([nidx[s] for s in keep], int)
            kidx = {s: i for i, s in enumerate(keep)}
            maf_idx = np.array([kidx[s] for s in maf_samples if s in kidx], int)
            nS = len(names)
            log(f"  VCF {chrom}: {nS} samples in file, keeping {len(keep)}; MAF computed on {len(maf_idx)}")
            continue
        n_lines += 1
        if n_lines % 250_000 == 0:
            print(f"   ... {n_lines:,} records, {n_win:,} in windows, {n_snv:,} kept ({time.time()-t0:.0f}s)", flush=True)
        t = line.split(b"\t", 9)
        if t[0] not in cb: continue
        pos = int(t[1])
        if pos > last_end: break
        i = bisect.bisect_right(starts, pos) - 1
        if i < 0 or pos > ends[i]: continue
        n_win += 1
        ref, alt = t[3], t[4]
        if len(ref) != 1 or len(alt) != 1 or alt not in (b"A", b"C", b"G", b"T") or ref not in (b"A", b"C", b"G", b"T"):
            continue
        blob = t[9].rstrip(b"\r\n")
        if t[8] == b"GT" and len(blob) == 4 * nS - 1:
            arr = np.frombuffer(blob, dtype=np.uint8)
            sep = arr[sel * 4 + 1]
            a0 = arr[sel * 4].astype(np.int16) - 48; a1 = arr[sel * 4 + 2].astype(np.int16) - 48
            if (sep != 124).any() or (a0 < 0).any() or (a0 > 1).any() or (a1 < 0).any() or (a1 > 1).any(): continue
        else:
            n_slow += 1
            f = blob.split(b"\t")
            g = [f[j].split(b":")[0] for j in sel]
            if any(len(x) != 3 or x[1:2] != b"|" or x[0:1] not in (b"0", b"1") or x[2:3] not in (b"0", b"1") for x in g):
                continue
            a0 = np.array([x[0] - 48 for x in g], np.int16); a1 = np.array([x[2] - 48 for x in g], np.int16)
        af = (a0[maf_idx].sum() + a1[maf_idx].sum()) / (2 * len(maf_idx))
        if min(af, 1 - af) < maf_min: continue
        n_snv += 1
        POS.append(pos); ID.append(t[2].decode()); REF.append(ref.decode()); ALT.append(alt.decode())
        H0.append(a0.astype(np.int8)); H1.append(a1.astype(np.int8))
    log(f"  VCF {chrom}: {n_lines:,} records scanned, {n_win:,} in windows, {n_snv:,} biallelic phased SNVs kept "
        f"(slow-path {n_slow}) in {time.time()-t0:.0f}s")
    H = np.stack([np.stack(H0), np.stack(H1)], -1) if POS else np.zeros((0, len(keep), 2), np.int8)
    var = pd.DataFrame(dict(chrom=chrom, pos=np.array(POS, int), rsid=ID, ref=REF, alt=ALT))
    return var, H, keep

VAR_L, HAP_L, VCF_SAMPLES = [], [], None
VCF_N_SAMPLES_IN_FILE = None
for ch in CFG["CHROMS"]:
    g_ch = GENES[GENES.chrom == ch]
    if len(g_ch) == 0: continue
    p = fetch(vcf_name(ch), vcf_urls(ch), min_bytes=50_000_000 if not CFG["SELF_TEST"] else 100,
              desc=f"1000G 30x phased {ch}",
              resolvers=[lambda ch=ch: crawl_listing([f"{b}/{HC}/working" for b in G1K[:3]], vcf_name(ch),
                                                     rf"phased.*\.{ch}\..*\.vcf\.gz$|\.{ch}\..*phased.*\.vcf\.gz$",
                                                     max_depth=1)])
    s, e = merged_windows(g_ch.tss.values, CFG["CIS_WINDOW"])
    maf_samples = [x for x in GEU.columns]
    v, H, keep = parse_vcf_windows(p, ch, s, e, sorted(vcf_keep), maf_samples, CFG["MAF_MIN"])
    if VCF_SAMPLES is None: VCF_SAMPLES = keep
    assert keep == VCF_SAMPLES, "sample order differs between chromosome files"
    VAR_L.append(v); HAP_L.append(H)
VAR = pd.concat(VAR_L, ignore_index=True)
HAP = np.concatenate(HAP_L, 0)              # n_var x n_vcf_samples x 2  (int8)
del HAP_L; gc.collect()
VAR["vid"] = VAR.chrom + ":" + VAR.pos.astype(str) + ":" + VAR.ref + ":" + VAR.alt
VAR["ccre"] = False
for ch in VAR.chrom.unique():
    m = (VAR.chrom == ch).values; VAR.loc[m, "ccre"] = in_ccre(ch, VAR.pos.values[m])
VSIDX = {s: i for i, s in enumerate(VCF_SAMPLES)}
log(f"Variants kept: {len(VAR):,} | HAP array {HAP.shape} ({HAP.nbytes/1e6:.0f} MB) | cCRE-overlapping: {VAR.ccre.mean():.3f}")

## 4. Module 1 — Parental haplotype reconstruction, Mendelian QC, origin assignment

In [ ]:
def origin_from_trio(child, father, mother):
    """Returns per-variant arrays for one trio: hap0_is_paternal (+1/-1/0 uninformative), mendel_error flags."""
    c, f, m = HAP[:, VSIDX[child]], HAP[:, VSIDX[father]], HAP[:, VSIDX[mother]]
    cd, fd, md = c.sum(1), f.sum(1), m.sum(1)
    pat = np.where(fd == 2, 1, np.where(fd == 0, 0, -1)); mat = np.where(md == 2, 1, np.where(md == 0, 0, -1))
    err = ((pat >= 0) & (mat >= 0) & (cd != pat + mat)) | ((pat == 0) & (cd == 2)) | ((pat == 1) & (cd == 0)) \
          | ((mat == 0) & (cd == 2)) | ((mat == 1) & (cd == 0))
    het = c[:, 0] != c[:, 1]
    alt_on_pat = np.where(pat >= 0, pat == 1, mat == 0)
    inf = het & ((pat >= 0) | (mat >= 0)) & ~err
    vote = np.zeros(len(cd), np.int8)
    vote[inf] = np.where((c[inf, 0] == 1) == alt_on_pat[inf], 1, -1)
    return vote, err

ORIGIN_QC, VOTES = [], {}
for child, r in qc_trios.iterrows():
    if not all(x in VSIDX for x in (child, r.father, r.mother)): continue
    vote, err = origin_from_trio(child, r.father, r.mother)
    VOTES[child] = vote
    inf = np.where(vote != 0)[0]
    seq = vote[inf]
    sw = 0.0
    for ch in VAR.chrom.unique():
        ii = inf[VAR.chrom.values[inf] == ch]
        if len(ii) > 1: sw += (np.diff(vote[ii]) != 0).sum()
    ORIGIN_QC.append(dict(child=child, father=r.father, mother=r.mother,
                          pop=PED.loc[child, "pop"] if child in PED.index else None,
                          in_geuvadis=child in geu_ids, in_mage=child in mage_ids,
                          n_variants=len(vote), n_informative=len(inf),
                          frac_hap0_paternal=float((seq == 1).mean()) if len(seq) else np.nan,
                          switch_rate=float(sw / max(len(inf) - 1, 1)),
                          mendel_error_rate=float(err.mean())))
ORIGIN_QC = pd.DataFrame(ORIGIN_QC)
if len(ORIGIN_QC):
    save_table(ORIGIN_QC, "S1b_trio_origin_phasing_qc")
    log(f"Origin QC on {len(ORIGIN_QC)} trios: median informative sites {ORIGIN_QC.n_informative.median():.0f}, "
        f"median frac(hap0=paternal) {ORIGIN_QC.frac_hap0_paternal.median():.3f}, "
        f"median switch rate {ORIGIN_QC.switch_rate.median():.4f}, "
        f"median Mendelian error {ORIGIN_QC.mendel_error_rate.median():.5f}")

## 5. Analysis cohort, per-gene cis windows and leakage-safe preprocessing

In [ ]:
# ---- analysis individuals: GEUVADIS ∩ VCF ----
S = [s for s in GEU.columns if s in VSIDX]
nS = len(S)
S_VIDX = np.array([VSIDX[s] for s in S])
POP = np.array([PED.loc[s, "pop"] if s in PED.index else "UNK" for s in S])
SUPER = np.array([PED.loc[s, "superpop"] if s in PED.index else "UNK" for s in S])
SEX = np.array([float(PED.loc[s, "sex"]) if s in PED.index and str(PED.loc[s, "sex"]).isdigit() else 1.5 for s in S])
FAM = np.array([FAMILY.get(s, s) for s in S])
log(f"Analysis cohort: {nS} individuals | populations: {dict(zip(*np.unique(POP, return_counts=True)))} | "
    f"families: {len(set(FAM))}")

# ---- per-gene cis variants ----
W = CFG["CIS_WINDOW"]
gene_rows, GENE_VARS = [], []
for gid, r in GENES.iterrows():
    m = np.where((VAR.chrom.values == r.chrom) & (np.abs(VAR.pos.values - r.tss) <= W))[0]
    if len(m) < CFG["MIN_VARIANTS_PER_GENE"]: continue
    gene_rows.append(gid); GENE_VARS.append(m)
GENES = GENES.loc[gene_rows].copy()
if CFG["MAX_GENES"]:
    keep_g = GENES.index[:CFG["MAX_GENES"]]
    GENE_VARS = [v for g, v in zip(GENES.index, GENE_VARS) if g in set(keep_g)]; GENES = GENES.loc[keep_g]
G = len(GENES)
GENES["n_cis_variants"] = [len(v) for v in GENE_VARS]
GENES["block"] = (GENES.chrom + "_" + (GENES.tss // 2_000_000).astype(str)).values
log(f"Genes with >= {CFG['MIN_VARIANTS_PER_GENE']} cis SNVs: {G} | median cis SNVs/gene: {GENES.n_cis_variants.median():.0f}")

# per-gene haplotypes for the analysis cohort (n_S x p x 2) and variant metadata
H_GENE, VMETA = [], []
_af = HAP[:, S_VIDX, :].mean(axis=(1, 2)); MAF = np.minimum(_af, 1 - _af)
for gi, (gid, r) in enumerate(GENES.iterrows()):
    idx = GENE_VARS[gi]
    H_GENE.append(np.ascontiguousarray(HAP[idx][:, S_VIDX, :].transpose(1, 0, 2)))
    sgn = 1 if r.strand == "+" else -1
    VMETA.append(np.column_stack([(VAR.pos.values[idx] - r.tss) * sgn / W, MAF[idx], VAR.ccre.values[idx].astype(float)]))
Y_RAW = GEU.loc[GENES.index, S].T.values.astype(float)          # n_S x G
Y_RAW = np.where(np.isnan(Y_RAW), np.nanmedian(Y_RAW, 0, keepdims=True), Y_RAW)
EXPR_PC_SRC = GEU[S].values.T.astype(np.float32)
EXPR_PC_SRC = EXPR_PC_SRC[:, np.nanstd(EXPR_PC_SRC, 0) > 0]
EXPR_PC_SRC = np.where(np.isnan(EXPR_PC_SRC), 0, EXPR_PC_SRC)
rng_pc = np.random.default_rng(0)
pc_var = rng_pc.choice(len(VAR), min(CFG["N_GENO_PC_VARIANTS"], len(VAR)), replace=False)
GENO_PC_SRC = HAP[pc_var][:, S_VIDX, :].sum(-1).T.astype(np.float32)
POP_LEVELS = sorted(set(POP))

def int_train_test(ytr, yte):
    n = len(ytr)
    z = norm.ppf((rankdata(ytr) - 0.5) / n)
    u, inv = np.unique(ytr, return_inverse=True)
    zm = np.bincount(inv, z) / np.bincount(inv)
    return z, (np.interp(yte, u, zm) if len(yte) else np.zeros(0))

def _pcs(X, tr, te, k):
    mu = X[tr].mean(0); sd = X[tr].std(0); sd[sd == 0] = 1
    A = (X[tr] - mu) / sd
    k = min(k, A.shape[0] - 2, A.shape[1])
    U, sv, Vt = np.linalg.svd(A, full_matrices=False)
    V = Vt[:k].T
    return A @ V, ((X[te] - mu) / sd) @ V if len(te) else np.zeros((0, k))

def prep_fold(tr, te, use_pop=True):
    """INT + covariate residualisation fit on TRAIN only, applied to TEST. Returns (R_tr, R_te) n x G."""
    tr = np.asarray(tr); te = np.asarray(te, int)
    Ytr = np.empty((len(tr), G)); Yte = np.empty((len(te), G))
    for g in range(G):
        Ytr[:, g], Yte[:, g] = int_train_test(Y_RAW[tr, g], Y_RAW[te, g])
    gtr, gte = _pcs(GENO_PC_SRC, tr, te, CFG["N_GENO_PCS"])
    etr, ete = _pcs(EXPR_PC_SRC, tr, te, CFG["N_EXPR_PCS"])
    def cov(ix, gp, ep):
        cols = [gp, ep, SEX[ix, None]]
        if use_pop:
            cols.append(np.stack([(POP[ix] == p).astype(float) for p in POP_LEVELS[1:]], 1) if len(POP_LEVELS) > 1
                        else np.zeros((len(ix), 0)))
        return np.hstack(cols)
    Ctr, Cte = cov(tr, gtr, etr), cov(te, gte, ete)
    mu, sd = Ctr.mean(0), Ctr.std(0); sd[sd == 0] = 1
    Xtr = np.hstack([np.ones((len(tr), 1)), (Ctr - mu) / sd]); Xte = np.hstack([np.ones((len(te), 1)), (Cte - mu) / sd])
    B = np.linalg.lstsq(Xtr, Ytr, rcond=None)[0]
    Rtr = Ytr - Xtr @ B; Rte = Yte - Xte @ B
    s = Rtr.std(0); s[s == 0] = 1
    return Rtr / s, Rte / s

def make_folds(groups, strata, k, seed=0):
    rng = np.random.default_rng(seed)
    fams = pd.DataFrame({"g": groups, "s": strata}).groupby("g").agg(s=("s", "first"), n=("s", "size")).reset_index()
    fold_of, load = {}, np.zeros(k)
    for _, sub in fams.groupby("s"):
        sub = sub.sample(frac=1, random_state=int(rng.integers(1e9))).sort_values("n", ascending=False, kind="stable")
        loc = np.zeros(k)
        for _, r in sub.iterrows():
            j = int(np.lexsort((load, loc))[0]); fold_of[r.g] = j; loc[j] += r.n; load[j] += r.n
    return np.array([fold_of[g] for g in groups])

FOLDS = make_folds(FAM, POP, CFG["N_FOLDS"], seed=2026)
log("Split A fold sizes:", np.bincount(FOLDS).tolist(), "| no family spans two folds:",
    all(len(set(FOLDS[FAM == f])) == 1 for f in set(FAM)))

# ---- origin orientation per (individual, gene): +1 hap0 paternal, -1 hap1 paternal, 0 unknown ----
ORIENT = np.zeros((nS, G), np.int8)
for i, s in enumerate(S):
    if s not in VOTES: continue
    v = VOTES[s]
    for gi in range(G):
        vv = v[GENE_VARS[gi]]; vv = vv[vv != 0]
        if len(vv) >= 3:
            fr = (vv == 1).mean()
            if fr >= 0.9: ORIENT[i, gi] = 1
            elif fr <= 0.1: ORIENT[i, gi] = -1
N_ORIGIN_IND = int((np.abs(ORIENT).sum(1) > 0).sum())
log(f"Individuals in the expression cohort with verified parental origin: {N_ORIGIN_IND} "
    f"({int((ORIENT != 0).sum())} individual-gene pairs)")

## 6. Per-gene baselines (Ridge, PrediXcan-style EN, pairwise cis/trans regression, XGBoost) + aFC-n-style model

In [ ]:
ALPHAS = np.logspace(-1, 4, 16)
def _std_fit(Xtr, Xte):
    m = Xtr.mean(0); s = Xtr.std(0); s[s < 1e-8] = 1
    return (Xtr - m) / s, (Xte - m) / s
def ridge_fp(Xtr, ytr, Xte):
    A, B = _std_fit(Xtr, Xte)
    mdl = RidgeCV(alphas=ALPHAS).fit(A, ytr)
    return (mdl.predict(B) if len(B) else np.zeros(0)), mdl
def ridge_oof(Xtr, ytr, groups):
    oof = np.zeros(len(ytr))
    ng = len(set(groups)); k = min(5, ng)
    for a, b in GroupKFold(k).split(Xtr, ytr, groups):
        oof[b] = ridge_fp(Xtr[a], ytr[a], Xtr[b])[0]
    return oof

_EN_SIG = inspect.signature(ElasticNetCV.__init__).parameters
def make_en(cv):
    kw = dict(l1_ratio=0.5, cv=cv, max_iter=3000, tol=1e-3, selection="random", random_state=0)
    if "n_alphas" in _EN_SIG and not ("alphas" in _EN_SIG and isinstance(_EN_SIG["alphas"].default, int)):
        kw["n_alphas"] = 20
    else:
        kw["alphas"] = 20
    return ElasticNetCV(**kw)

_LA_SIG = inspect.signature(LassoCV.__init__).parameters
def make_lasso(cv):
    kw = dict(cv=cv, max_iter=5000, tol=1e-4, random_state=0, selection="cyclic")
    if "n_alphas" in _LA_SIG and not ("alphas" in _LA_SIG and isinstance(_LA_SIG["alphas"].default, int)):
        kw["n_alphas"] = 30
    else:
        kw["alphas"] = 30
    return LassoCV(**kw)

def lasso_fit_predict(Ftr, rtr, Fte, groups):
    """Sparse interaction head on additive residuals. Returns test prediction and RAW-scale coefficients."""
    m = Ftr.mean(0); sd = Ftr.std(0); ok = sd > 1e-8
    coef = np.zeros(Ftr.shape[1])
    if ok.sum() == 0 or len(set(groups)) < 3: return np.zeros(len(Fte)), coef
    A = (Ftr[:, ok] - m[ok]) / sd[ok]; B = (Fte[:, ok] - m[ok]) / sd[ok]
    try:
        cv = list(GroupKFold(min(5, len(set(groups)))).split(A, rtr, groups))
        las = make_lasso(cv).fit(A, rtr)
    except Exception:
        return np.zeros(len(Fte)), coef, None
    coef[ok] = las.coef_ / sd[ok]
    return (las.predict(B) if len(B) else np.zeros(0)), coef, float(las.alpha_)

def lasso_crossfit(Ftr, rtr, groups, alpha, k=5):
    """Out-of-fold predictions with the penalty tuned once on all training rows (fixed-alpha refits: ~5x faster)."""
    from sklearn.linear_model import Lasso
    oof = np.zeros(len(rtr))
    if alpha is None: return oof
    for a, b in GroupKFold(min(k, len(set(groups)))).split(Ftr, rtr, groups):
        m = Ftr[a].mean(0); sd = Ftr[a].std(0); ok = sd > 1e-8
        if ok.sum() == 0: continue
        las = Lasso(alpha=alpha, max_iter=5000, tol=1e-3, selection="random", random_state=0)
        las.fit((Ftr[a][:, ok] - m[ok]) / sd[ok], rtr[a])
        oof[b] = las.predict((Ftr[b][:, ok] - m[ok]) / sd[ok])
    return oof

def clump(Z, r, ok, K, thr):
    order = np.argsort(-np.abs(r)); chosen = []
    n = Z.shape[0]
    for j in order:
        if not ok[j]: continue
        if chosen:
            rr = (Z[:, chosen].T @ Z[:, j]) / n
            if np.max(rr ** 2) >= thr: continue
        chosen.append(j)
        if len(chosen) >= K: break
    return np.array(chosen, int)

def random_matched(maf, ok, sel, rng):
    pool = np.setdiff1d(np.where(ok)[0], sel); out = []
    for j in sel:
        if len(pool) == 0: break
        cand = pool[np.argsort(np.abs(maf[pool] - maf[j]))[:5]]
        c = int(rng.choice(cand)); out.append(c); pool = pool[pool != c]
    return np.array(out, int)

PAIRS_CACHE = {}
def pair_idx(k):
    if k not in PAIRS_CACHE:
        p = list(itertools.combinations(range(k), 2))
        PAIRS_CACHE[k] = (np.array([a for a, b in p], int), np.array([b for a, b in p], int))
    return PAIRS_CACHE[k]

def pair_features(hA, hB, kind):
    d = hA + hB; dom = hA * hB
    i, j = pair_idx(hA.shape[1]); f = [dom]
    if len(i):
        if "cis" in kind:   f.append(hA[:, i] * hA[:, j] + hB[:, i] * hB[:, j])
        if "trans" in kind: f.append(hA[:, i] * hB[:, j] + hB[:, i] * hA[:, j])
        if "blind" in kind: f.append(d[:, i] * d[:, j])
    return np.hstack(f)

# Variant features are LABEL-FREE (no marginal r / coefficients): label-derived features computed on the same
# training individuals leak into the residual learner (stacking leakage) and make it overfit.
NF = 6  # [selection rank, signed TSS distance, |distance|, MAF, ENCODE cCRE, promoter (<2 kb)]
CTX_DIMS = [1, 2, 3, 4, 5]
def variant_features(sel, vm):
    k = len(sel)
    Z = np.zeros((k, NF), np.float32)
    if k == 0: return Z
    Z[:, 0] = np.arange(k) / max(k, 1)
    Z[:, 1] = vm[sel, 0]; Z[:, 2] = np.abs(vm[sel, 0]); Z[:, 3] = vm[sel, 1] * 2
    Z[:, 4] = vm[sel, 2]; Z[:, 5] = (np.abs(vm[sel, 0]) * W < 2000).astype(float)
    return Z

def en_fit_predict(Xtr, ytr, Xte, groups, n_cv=3):
    A, B = _std_fit(Xtr, Xte)
    try:
        cv = list(GroupKFold(min(n_cv, len(set(groups)))).split(A, ytr, groups))
        en = make_en(cv).fit(A, ytr)
        return (en.predict(B) if len(B) else np.zeros(0)), en
    except Exception:
        return np.full(len(B), ytr.mean()), None

def en_crossfit(Xtr, ytr, groups, en, k=5):
    """Cross-fitting: out-of-fold EN predictions for the training rows, refitting with the penalty that was tuned
       once on all training rows (fixed-alpha refits are ~5x faster than nested CV)."""
    from sklearn.linear_model import ElasticNet
    oof = np.zeros(len(ytr))
    if en is None: return oof + ytr.mean()
    for a, b in GroupKFold(min(k, len(set(groups)))).split(Xtr, ytr, groups):
        A, B = _std_fit(Xtr[a], Xtr[b])
        m = ElasticNet(alpha=en.alpha_, l1_ratio=en.l1_ratio, max_iter=3000, tol=1e-3, selection="random",
                       random_state=0).fit(A, ytr[a])
        oof[b] = m.predict(B)
    return oof

def fit_gene(gi, Hg, vm, tr, te, ytr, yte, groups_tr, seed, models, K, KP,
             heads=("cis", "trans", "cis+trans", "shuffle", "random")):
    warnings.filterwarnings("ignore")
    rng = np.random.default_rng(seed * 7919 + gi)
    hA = Hg[:, :, 0].astype(np.float32); hB = Hg[:, :, 1].astype(np.float32); D = hA + hB
    Dtr = D[tr]; mu = Dtr.mean(0); sd = Dtr.std(0); ok = sd > 1e-6
    Ztr = np.zeros_like(Dtr); Ztr[:, ok] = (Dtr[:, ok] - mu[ok]) / sd[ok]
    yc = (ytr - ytr.mean()) / (ytr.std() + 1e-12)
    r = Ztr.T @ yc / len(tr); r[~ok] = 0
    sel = clump(Ztr, r, ok, K, CFG["CLUMP_R2"])
    out = dict(gi=gi, sel=sel, preds={})
    if len(sel) == 0:
        out.update(base_tr=np.zeros(len(tr)), base_te=np.zeros(len(te)), base_res_var=1.0,
                   z_sel=np.zeros((0, NF), np.float32), selr=np.array([], int), z_rand=np.zeros((0, NF), np.float32))
        for m in BASE_NAMES: out["preds"][m] = np.zeros(len(te))
        out["int"] = {k_: dict(tr=np.zeros(len(tr)), te=np.zeros(len(te)), coef=None, k=0)
                      for k_ in ["cis", "trans", "cis+trans", "shuffle", "random"]}
        return out
    Xtr, Xte = D[tr][:, sel], D[te][:, sel]
    p_ridge, mdl = ridge_fp(Xtr, ytr, Xte)
    out["preds"]["Ridge (additive, clumped top-K)"] = p_ridge
    # strongest additive model = PrediXcan-style ElasticNet over up to EN_MAX_VARIANTS cis SNVs (ranked on TRAIN only).
    # It is the BASE of OriginSib-CompNet (cross-fitted, see en_crossfit) -> the network only learns what additive EN
    # cannot explain. The reported EN baseline row is the standard EN refit on ALL training individuals.
    cols = np.argsort(-np.abs(r))[:CFG["EN_MAX_VARIANTS"]]; cols = cols[ok[cols]]
    p_en, en_m = en_fit_predict(D[tr][:, cols], ytr, D[te][:, cols], groups_tr)
    base_tr = en_crossfit(D[tr][:, cols], ytr, groups_tr, en_m)
    out["preds"]["ElasticNet (PrediXcan-style)"] = p_en
    out.update(base_tr=base_tr, base_te=p_en, base_res_var=float(np.var(ytr - base_tr)), ridge_alpha=float(mdl.alpha_),
               en_cols=cols)
    out["z_sel"] = variant_features(sel, vm)
    selr = random_matched(vm[:, 1], ok, sel, rng)
    out["selr"] = selr
    out["z_rand"] = variant_features(selr, vm)
    kp = sel[:KP]
    # ---- OriginSib stage 2: sparse phase-aware interaction head on cross-fitted EN residuals ----
    #   features = [dominance hA_i*hB_i | cis hA_i hA_j + hB_i hB_j | trans hA_i hB_j + hB_i hA_j]  (top-KP variants)
    #   variants: cis-only (A1), trans-only (A2), cis+trans (A3-A6), shuffled phase (A7), random matched variants (A8)
    r_oof = ytr - base_tr
    swap = (rng.random(hA.shape) < 0.5) & (hA != hB)
    hA_s, hB_s = np.where(swap, hB, hA), np.where(swap, hA, hB)
    INT = {}
    specs = {"cis": (hA, hB, kp, "cis"), "trans": (hA, hB, kp, "trans"), "cis+trans": (hA, hB, kp, "cis+trans"),
             "shuffle": (hA_s, hB_s, kp, "cis+trans"), "random": (hA, hB, selr[:KP], "cis+trans")}
    for key, (a_, b_, idx, kind) in specs.items():
        if key not in heads: continue
        if len(idx) < 2:
            INT[key] = dict(tr=np.zeros(len(tr)), te=np.zeros(len(te)), coef=None, k=len(idx)); continue
        F = pair_features(a_[:, idx], b_[:, idx], kind)
        p_te, coef, alpha = lasso_fit_predict(F[tr], r_oof, F[te], groups_tr)
        p_tr = lasso_crossfit(F[tr], r_oof, groups_tr, alpha) if key in ("cis+trans", "shuffle", "random") else None
        INT[key] = dict(tr=p_tr, te=p_te, coef=coef, k=len(idx))
    out["int"] = INT
    out["preds"]["OriginSib stage 1+2 (EN + sparse cis/trans)"] = p_en + INT["cis+trans"]["te"]
    if "PAIR" in models:
        for kind, name in [("blind", "Pairwise SNPxSNP (phase-blind)"), ("cis", "Pairwise + cis"),
                           ("trans", "Pairwise + trans"), ("cis+trans", "Pairwise + cis + trans")]:
            Ftr = np.hstack([Xtr, pair_features(hA[tr][:, kp], hB[tr][:, kp], kind)])
            Fte = np.hstack([Xte, pair_features(hA[te][:, kp], hB[te][:, kp], kind)])
            out["preds"][name] = ridge_fp(Ftr, ytr, Fte)[0]
    if "XGB" in models:
        XA = np.hstack([hA[:, sel], hB[:, sel]]); XB = np.hstack([hB[:, sel], hA[:, sel]])
        mdl_x = xgb.XGBRegressor(n_estimators=300, max_depth=3, learning_rate=0.05, subsample=0.8,
                                 colsample_bytree=0.8, min_child_weight=5, reg_lambda=1.0, n_jobs=1,
                                 tree_method="hist", random_state=seed, verbosity=0)
        mdl_x.fit(np.vstack([XA[tr], XB[tr]]), np.concatenate([ytr, ytr]))
        out["preds"]["XGBoost (A/B-symmetrised)"] = 0.5 * (mdl_x.predict(XA[te]) + mdl_x.predict(XB[te]))
    return out

def run_gene_models(tr, te, Rtr, Rte, seed, models=("PAIR", "XGB"), genes=None,
                    heads=("cis", "trans", "cis+trans", "shuffle", "random")):
    genes = range(G) if genes is None else genes
    groups_tr = FAM[tr]
    res = Parallel(n_jobs=N_JOBS, backend="loky", batch_size=4)(
        delayed(fit_gene)(g, H_GENE[g], VMETA[g], tr, te, Rtr[:, g], Rte[:, g], groups_tr, seed, models,
                          CFG["K_VARIANTS"], CFG["K_PAIR"], heads) for g in genes)
    return {r["gi"]: r for r in res}

def assemble_tensors(res, K=None):
    """Pad selected variants to K: HA/HB (G x nS x K), mask, Z (G x K x F), random-variant counterparts."""
    K = K or CFG["K_VARIANTS"]
    HA = np.zeros((G, nS, K), np.int8); HB = np.zeros_like(HA); M = np.zeros((G, K), np.float32)
    Z = np.zeros((G, K, NF), np.float32)
    HAr = np.zeros_like(HA); HBr = np.zeros_like(HA); Mr = np.zeros_like(M); Zr = np.zeros_like(Z)
    for g, r in res.items():
        k = len(r["sel"])
        if k:
            HA[g, :, :k] = H_GENE[g][:, r["sel"], 0]; HB[g, :, :k] = H_GENE[g][:, r["sel"], 1]
            M[g, :k] = 1; Z[g, :k] = r["z_sel"]
        kr = len(r.get("selr", []))
        if kr:
            HAr[g, :, :kr] = H_GENE[g][:, r["selr"], 0]; HBr[g, :, :kr] = H_GENE[g][:, r["selr"], 1]
            Mr[g, :kr] = 1; Zr[g, :kr] = r["z_rand"]
    return dict(HA=HA, HB=HB, M=M, Z=Z, HAr=HAr, HBr=HBr, Mr=Mr, Zr=Zr)

def fit_afcn(HA, HB, M, Y, tr, te, groups_tr, kp=None, steps=500, lams=(1e-3, 1e-2, 1e-1, 1.0)):
    """aFC-n-style haplotype model (functional form of Mohammadi et al. aFC-n) on the analysis scale, all genes batched:
         y = b0 + c * log2( (2^{sum_i b_i hA_i} + 2^{sum_i b_i hB_i}) / 2 )
       on the top-kp independent (clumped) cis variants. The L2 penalty is chosen PER GENE on an inner validation
       split of training families, then the model is refit on all training individuals."""
    kp = kp or CFG["K_PAIR"]; dev = DEVICE; L2 = math.log(2)
    HA, HB, M = HA[..., :kp], HB[..., :kp], M[:, :kp]
    Yfull = torch.tensor(Y, dtype=torch.float32, device=dev)
    def fit(idx_rows, lam_vec, yt):
        hA = torch.tensor(HA[:, idx_rows], dtype=torch.float32, device=dev)
        hB = torch.tensor(HB[:, idx_rows], dtype=torch.float32, device=dev)
        m = torch.tensor(M, device=dev)[:, None, :]; Gn, _, K = hA.shape
        beta = torch.zeros(Gn, K, device=dev, requires_grad=True)
        b0 = torch.zeros(Gn, 1, device=dev, requires_grad=True); lc = torch.zeros(Gn, 1, device=dev, requires_grad=True)
        opt = torch.optim.Adam([beta, b0, lc], lr=0.05)
        lam_t = torch.tensor(lam_vec, dtype=torch.float32, device=dev)
        def f(a, b):
            sA = (a * m * beta[:, None, :]).sum(-1).clamp(-20, 20); sB = (b * m * beta[:, None, :]).sum(-1).clamp(-20, 20)
            return b0 + torch.exp(lc) * (torch.logaddexp(sA * L2, sB * L2) / L2 - 1.0)
        for _ in range(steps):
            opt.zero_grad()
            loss = (((f(hA, hB) - yt) ** 2).mean(1) + lam_t * (beta ** 2).sum(1)).sum()
            loss.backward(); opt.step()
        def pred(rows):
            with torch.no_grad():
                return f(torch.tensor(HA[:, rows], dtype=torch.float32, device=dev),
                         torch.tensor(HB[:, rows], dtype=torch.float32, device=dev)).cpu().numpy()
        return pred
    # inner validation split by family (TRAIN only); Y columns are ordered like tr
    pos = {v: i for i, v in enumerate(tr)}
    fams = np.array(sorted(set(groups_tr))); rng = np.random.default_rng(0)
    vf = set(rng.choice(fams, max(1, len(fams) // 5), replace=False))
    va = np.array([i for i in tr if groups_tr[pos[i]] in vf]); fi = np.array([i for i in tr if groups_tr[pos[i]] not in vf])
    yfi = Yfull[:, [pos[i] for i in fi]]; yva = Y[:, [pos[i] for i in va]]
    G_ = HA.shape[0]; best = np.full(G_, np.inf); best_lam = np.full(G_, lams[0])
    for lam in lams:
        pv = fit(fi, np.full(G_, lam), yfi)(va)
        mse = ((pv - yva) ** 2).mean(1)
        better = mse < best; best[better] = mse[better]; best_lam[better] = lam
    return fit(tr, best_lam, Yfull)(te)     # G x n_te

## 7. OriginSib-CompNet

Three stages, all fit on training families only. The blueprint (§7) recommends *"sparse interaction regression + small MLP/DeepSets encoder"* for a first prototype.

1. **Additive**: PrediXcan-style ElasticNet on ≤500 cis SNVs. Cross-fitted: training rows get out-of-fold predictions.
2. **Sparse phase-aware interactions** (blueprint Modules 3–4) on stage-1 residuals, using Lasso with family-grouped CV:
   $C_g=\sum_{i<j}\gamma_{gij}(h_{Ai}h_{Aj}+h_{Bi}h_{Bj})$ (cis), $T_g=\sum_{i,j}\theta_{gij}h_{Ai}h_{Bj}$ (symmetric trans), plus dominance. The coefficients are interpretable.
3. **Neural residual**: DeepSets haplotype encoder with gene-specific and annotation-based variant embeddings, non-linear cis and symmetric trans branches with explicit bilinear terms, an antisymmetric POE head that is active only for Mendelian-verified origin, and a heteroscedastic variance head. A **per-gene stacking gate**, fit on validation families, keeps the correction only where it helps.

In [ ]:
class MLP(nn.Module):
    def __init__(self, i, o, H, p):
        super().__init__()
        self.net = nn.Sequential(nn.Linear(i, H), nn.LayerNorm(H), nn.GELU(), nn.Dropout(p), nn.Linear(H, o))
    def forward(self, x): return self.net(x)

class OriginSibCompNet(nn.Module):
    """Residual learner on top of the per-gene additive baseline.
       variant embedding e_i = MLP(z_i) (gene-agnostic -> transferable to unseen genes)
       phi(h) = sum_i h_i e_i / sqrt(K)  (DeepSets over the alt alleles of ONE haplotype)
       additive : sum_i w(z_i) (hA_i + hB_i)
       cis      : g(phi(hA)) + g(phi(hB))                  -- same-haplotype non-additivity
       trans    : t(phi(hA)*phi(hB), |phi(hA)-phi(hB)|)    -- symmetric in A<->B (no artificial origin effect)
       POE      : [q(phi_pat - phi_mat) - q(phi_mat - phi_pat)] * 1[origin verified]  (antisymmetric)
       variance : heteroscedastic log-variance head"""
    def __init__(self, F=NF, H=128, E=32, p=0.2, cis=True, trans=True, poe=False, hetero=True, K=32, G=1):
        super().__init__()
        self.cfg = dict(cis=cis, trans=trans, poe=poe, hetero=hetero); self.sK = math.sqrt(K); self.E = E
        self.emb = MLP(F, E, H, p); self.addw = MLP(F, 1, H, p)
        # gene-specific variant embeddings / additive weights, learned from TRAINING individuals only.
        # They start at 0, so a gene never seen in training (Split B) uses only the shared annotation embedding.
        self.U = nn.Parameter(torch.zeros(G, K, E)); self.u = nn.Parameter(torch.zeros(G, K))
        self.cis_f = MLP(E, 1, H, p) if cis else None
        self.tr_f = MLP(2 * E, 1, H, p) if trans else None
        self.poe_f = MLP(E, 1, H // 2, p) if poe else None
        self.var_f = MLP(2 * E, 1, H // 2, p) if hetero else None
        # explicit bilinear interaction terms (blueprint Modules 3-4), gene-specific through e_i = MLP(z_i) + U_g,i:
        #   cis   C_g = sum_ij gamma_gij (hA_i hA_j + hB_i hB_j),  gamma_g = e_g S_c e_g^T
        #   trans T_g = sum_ij theta_gij  hA_i hB_j,                theta_g = e_g S_t e_g^T, S_t symmetric (A<->B invariant)
        self.S_c = nn.Parameter(torch.zeros(E, E)) if cis else None
        self.S_t = nn.Parameter(torch.zeros(E, E)) if trans else None
        # zero-init every residual output layer: at initialisation the model == the additive EN base,
        # so early stopping can never return something worse (on validation) than the base.
        for m in [self.addw, self.cis_f, self.tr_f, self.poe_f, self.var_f]:
            if m is not None:
                nn.init.zeros_(m.net[-1].weight); nn.init.zeros_(m.net[-1].bias)
    def tables(self, Z, M):
        """Variant embeddings / additive weights depend only on the gene's variant features -> computed once per
           gene per step (G x K) and gathered per row; this is mathematically identical and ~10-50x faster."""
        return (self.emb(Z) + self.U) * M.unsqueeze(-1), (self.addw(Z).squeeze(-1) + self.u) * M
    def forward(self, hA, hB, e, w, orient):
        pA = (hA.unsqueeze(-1) * e).sum(1) / self.sK; pB = (hB.unsqueeze(-1) * e).sum(1) / self.sK
        add = (w * (hA + hB)).sum(1)
        z0 = torch.zeros_like(pA)
        cis = (self.cis_f(pA) + self.cis_f(pB) - 2 * self.cis_f(z0)).squeeze(-1) if self.cis_f is not None else torch.zeros_like(add)
        trn = (self.tr_f(torch.cat([pA * pB, (pA - pB).abs()], -1)) - self.tr_f(torch.cat([z0, z0], -1))).squeeze(-1) \
            if self.tr_f is not None else torch.zeros_like(add)
        if self.S_c is not None:
            Sc = 0.5 * (self.S_c + self.S_c.T)
            cis = cis + ((pA @ Sc) * pA).sum(-1) + ((pB @ Sc) * pB).sum(-1)
        if self.S_t is not None:
            St = 0.5 * (self.S_t + self.S_t.T)
            trn = trn + ((pA @ St) * pB).sum(-1)
        if self.poe_f is not None:
            o = orient.unsqueeze(-1).float()
            pat = torch.where(o > 0, pA, pB); mat = torch.where(o > 0, pB, pA); d = pat - mat
            poe = (self.poe_f(d) - self.poe_f(-d)).squeeze(-1) * (orient != 0).float()
        else:
            poe = torch.zeros_like(add)
        mu = add + cis + trn + poe
        lv = self.var_f(torch.cat([pA + pB, pA * pB], -1)).squeeze(-1).clamp(-5, 3) if self.var_f is not None \
            else torch.zeros_like(mu)
        return mu, lv, cis, trn

# OriginSib-CompNet = stage 1 additive EN  +  stage 2 sparse phase-aware cis/trans interaction head
#                     +  stage 3 gated neural DeepSets residual (non-linear cis/trans, POE head, heteroscedastic variance)
# "int" = which stage-2 head is used; "nn" = whether stage 3 is trained.
ABLATIONS = OrderedDict([
    ("A0 additive only",            dict(nn=False, int=None)),
    ("A1 additive + cis",           dict(nn=False, int="cis")),
    ("A2 additive + trans",         dict(nn=False, int="trans")),
    ("A3 cis+trans, no reg. context", dict(int="cis+trans", cis=True, trans=True, ctx=False, hetero=True, poe=False)),
    ("A4 cis+trans + context",      dict(int="cis+trans", cis=True, trans=True, ctx=True,  hetero=True,  poe=False)),
    ("A5 A4 without uncertainty",   dict(int="cis+trans", cis=True, trans=True, ctx=True,  hetero=False, poe=False)),
    ("A6 full OriginSib-CompNet",   dict(int="cis+trans", cis=True, trans=True, ctx=True,  hetero=True,  poe=True)),
    ("A7 full, shuffled phase (neg. ctrl)",  dict(int="shuffle", cis=True, trans=True, ctx=True, hetero=True, poe=True, shuffle=True)),
    ("A8 full, random matched variants (neg. ctrl)", dict(int="random", cis=True, trans=True, ctx=True, hetero=True, poe=True, random=True)),
])

def build_bases(res, tr, te):
    """Per stage-2 head: train rows = cross-fitted EN + cross-fitted interaction head; test rows = EN + head."""
    out = {}
    for key in [None, "cis", "trans", "cis+trans", "shuffle", "random"]:
        B = np.zeros((G, nS), np.float32)
        for g, r in res.items():
            B[g, te] = r["preds"]["ElasticNet (PrediXcan-style)"]; B[g, tr] = r["base_tr"]
            if key is not None and key in r["int"]:
                B[g, te] += r["int"][key]["te"]
                if r["int"][key]["tr"] is not None: B[g, tr] += r["int"][key]["tr"]
        out[key] = B
    return out
FULL = "A6 full OriginSib-CompNet"

class FoldTensors:
    def __init__(self, T, base, Y, orient):
        d = DEVICE
        self.HA = torch.tensor(T["HA"], device=d); self.HB = torch.tensor(T["HB"], device=d)
        self.M = torch.tensor(T["M"], device=d); self.Z = torch.tensor(T["Z"], device=d)
        self.HAr = torch.tensor(T["HAr"], device=d); self.HBr = torch.tensor(T["HBr"], device=d)
        self.Mr = torch.tensor(T["Mr"], device=d); self.Zr = torch.tensor(T["Zr"], device=d)
        self.base = torch.tensor(base, dtype=torch.float32, device=d); self.Y = torch.tensor(Y, dtype=torch.float32, device=d)
        self.O = torch.tensor(orient.T.copy(), device=d)          # G x nS

def _rows(genes, inds):
    g, i = np.meshgrid(np.asarray(genes), np.asarray(inds), indexing="ij")
    return torch.tensor(g.ravel(), device=DEVICE), torch.tensor(i.ravel(), device=DEVICE)

def _gene_inputs(FT, cfg):
    Z, M = (FT.Zr, FT.Mr) if cfg.get("random") else (FT.Z, FT.M)
    if not cfg.get("ctx", True):
        Z = Z.clone(); Z[..., CTX_DIMS] = 0
    return Z, M

def _get(FT, cfg, gi, ii, shuf=None):
    if cfg.get("random"):
        hA, hB = FT.HAr[gi, ii].float(), FT.HBr[gi, ii].float()
    else:
        hA, hB = FT.HA[gi, ii].float(), FT.HB[gi, ii].float()
    if shuf is not None:
        s = shuf[gi, ii]
        hA, hB = torch.where(s, hB, hA), torch.where(s, hA, hB)
    return hA, hB

def _fwd(net, FT, cfg, gi, ii, shuf, tabs=None):
    if tabs is None: tabs = net.tables(*_gene_inputs(FT, cfg))
    e, w = tabs
    hA, hB = _get(FT, cfg, gi, ii, shuf)
    return net(hA, hB, e[gi], w[gi], FT.O[gi, ii])

def train_compnet(FT, cfg, train_genes, fit_inds, val_inds, seed, lr, verbose=False):
    seed_all(seed)
    K = FT.HA.shape[-1]
    net = OriginSibCompNet(H=CFG["HIDDEN"], E=CFG["EMB"], p=CFG["DROPOUT"], cis=cfg["cis"], trans=cfg["trans"],
                           poe=cfg["poe"], hetero=cfg["hetero"], K=K, G=FT.HA.shape[0]).to(DEVICE)
    opt = torch.optim.AdamW(net.parameters(), lr=lr, weight_decay=CFG["WEIGHT_DECAY"])
    shuf = None
    if cfg.get("shuffle"):
        gen = torch.Generator(device="cpu").manual_seed(seed)
        shuf = (torch.rand(FT.HA.shape, generator=gen) < 0.5).to(DEVICE) & (FT.HA != FT.HB)
    gtr, itr = _rows(train_genes, fit_inds); gva, iva = _rows(train_genes, val_inds)
    def loss_fn(gi, ii, train=True, tabs=None):
        mu, lv, c, t = _fwd(net, FT, cfg, gi, ii, shuf, tabs)
        res = FT.Y[gi, ii] - FT.base[gi, ii] - mu
        L = (0.5 * (lv + res ** 2 * torch.exp(-lv))).mean() if cfg["hetero"] else (res ** 2).mean()
        if train:
            L = L + CFG["INTERACTION_L1"] * (c.abs().mean() + t.abs().mean()) \
                + CFG["GENE_L2"] * ((net.U[gi] ** 2).sum((1, 2)) + (net.u[gi] ** 2).sum(1)).mean()
        return L, (res ** 2).mean()
    best, best_state, bad, ep_used = 1e18, None, 0, 0
    n = len(gtr); B = CFG["BATCH"]
    for ep in range(CFG["MAX_EPOCHS"] + 1):
        if ep > 0:      # ep == 0 evaluates the untrained model (== additive base) as the reference state
            net.train(); perm = torch.randperm(n, device=DEVICE)
            for s in range(0, n, B):
                b = perm[s:s + B]; opt.zero_grad(); L, _ = loss_fn(gtr[b], itr[b]); L.backward()
                nn.utils.clip_grad_norm_(net.parameters(), 5.0); opt.step()
        net.eval()
        with torch.no_grad():
            vl = []; tb = net.tables(*_gene_inputs(FT, cfg))
            for s in range(0, len(gva), 8192):
                vl.append(loss_fn(gva[s:s + 8192], iva[s:s + 8192], train=False, tabs=tb)[0].item() * len(gva[s:s + 8192]))
            v = sum(vl) / max(len(gva), 1)
        ep_used = ep
        if v < best - CFG["MIN_DELTA"]:
            best, bad = v, 0; best_state = {k: x.detach().clone() for k, x in net.state_dict().items()}
        else:
            bad += 1
            if bad >= CFG["PATIENCE"]: break
    net.load_state_dict(best_state); net.eval()
    net._shuf = shuf; net._cfg = cfg; net._best_val = best; net._epochs = ep_used
    net._var_scale = 1.0
    # ---- per-gene stacking gate (fit on validation FAMILIES of the training set only) ----
    #   alpha_g = clip(<r,mu>/<mu,mu>, 0, 1) if the t-statistic of that regression > GATE_T else 0,
    #   r = y - additive base.  Genes where the learned correction does not help on validation fall back to EN.
    Gn = FT.HA.shape[0]
    with torch.no_grad():
        gg, ig = _rows(np.arange(Gn), val_inds)
        tb = net.tables(*_gene_inputs(FT, cfg)); MU = []
        for s in range(0, len(gg), 16384):
            MU.append(_fwd(net, FT, cfg, gg[s:s + 16384], ig[s:s + 16384], shuf, tb)[0])
        MU = torch.cat(MU).reshape(Gn, -1); Rv = (FT.Y[gg, ig] - FT.base[gg, ig]).reshape(Gn, -1)
        num = (Rv * MU).sum(1); den = (MU * MU).sum(1) + 1e-8; a = num / den
        nv = MU.shape[1]; sig2 = ((Rv - a[:, None] * MU) ** 2).sum(1) / max(nv - 1, 1)
        tstat = a * torch.sqrt(den / (sig2 + 1e-8))
        net._alpha = torch.where(tstat > CFG["GATE_T"], a.clamp(0, 1), torch.zeros_like(a))
        net._alpha_frac = float((net._alpha > 0).float().mean())
    with torch.no_grad():
        if not cfg["hetero"]:   # homoscedastic variance from validation residuals
            net._const_var = float(np.mean([loss_fn(gva[s:s+8192], iva[s:s+8192], False)[1].item()
                                             for s in range(0, len(gva), 8192)]))
        else:                   # post-hoc variance recalibration on validation families (train data only)
            tb = net.tables(*_gene_inputs(FT, cfg)); z2 = []
            for s in range(0, len(gva), 8192):
                g, i = gva[s:s + 8192], iva[s:s + 8192]
                mu, lv, _, _ = _fwd(net, FT, cfg, g, i, shuf, tb); mu = mu * net._alpha[g]
                z2.append(((FT.Y[g, i] - FT.base[g, i] - mu) ** 2 * torch.exp(-lv)).cpu())
            net._var_scale = float(torch.cat(z2).mean().clamp(0.25, 4.0))
    return net

@torch.no_grad()
def predict_compnet(net, FT, genes, inds):
    gi, ii = _rows(genes, inds); mus, vs = [], []
    tb = net.tables(*_gene_inputs(FT, net._cfg))
    for s in range(0, len(gi), 16384):
        g, i = gi[s:s + 16384], ii[s:s + 16384]
        mu, lv, _, _ = _fwd(net, FT, net._cfg, g, i, net._shuf, tb); mu = mu * net._alpha[g]
        mus.append((FT.base[g, i] + mu).cpu()); vs.append((torch.exp(lv) * net._var_scale).cpu() if net._cfg["hetero"]
                                                          else torch.full_like(mu, net._const_var).cpu())
    mu = torch.cat(mus).numpy().reshape(len(genes), len(inds)); v = torch.cat(vs).numpy().reshape(len(genes), len(inds))
    return mu, v

@torch.no_grad()
def counterfactual_contrasts(net, FT, g, inds, kp, n_bg=32, seed=0):
    """Model-implied interaction contrasts for all pairs among the first kp candidate variants of gene g (one batch).
       Background = observed haplotypes of held-out individuals, with both focal variants set to REF on both copies.
       eps_cis   = f(A:a+b) - f(A:a) - f(A:b) + f(0)      (both alt alleles on the same haplotype)
       eps_trans = f(A:a, B:b) - f(A:a) - f(B:b) + f(0)   (alt alleles on different haplotypes)"""
    rng = np.random.default_rng(seed)
    inds = rng.choice(inds, min(n_bg, len(inds)), replace=False)
    k = int(FT.M[g].sum().item()); kp = min(kp, k)
    if kp < 2: return []
    pi, pj = pair_idx(kp); P = len(pi); n = len(inds); K = FT.HA.shape[-1]
    gi = torch.full((n,), g, device=DEVICE); ii = torch.tensor(inds, device=DEVICE)
    hA0, hB0 = _get(FT, net._cfg, gi, ii, None)
    e, w = net.tables(*_gene_inputs(FT, net._cfg)); e, w = e[g:g + 1], w[g:g + 1]
    O = FT.O[gi, ii]
    Ea = torch.zeros(P, 1, K, device=DEVICE); Eb = torch.zeros(P, 1, K, device=DEVICE)
    Ea[torch.arange(P), 0, torch.tensor(pi)] = 1; Eb[torch.arange(P), 0, torch.tensor(pj)] = 1
    keep = 1 - Ea - Eb
    A0 = hA0[None] * keep; B0 = hB0[None] * keep                     # P x n x K
    cfgs = [(A0, B0), (A0 + Ea, B0), (A0 + Eb, B0), (A0, B0 + Eb), (A0 + Ea + Eb, B0), (A0 + Ea, B0 + Eb)]
    HA_ = torch.stack([c[0] for c in cfgs]).reshape(-1, K); HB_ = torch.stack([c[1] for c in cfgs]).reshape(-1, K)
    R = HA_.shape[0] // n
    mu = net(HA_, HB_, e.expand(HA_.shape[0], -1, -1), w.expand(HA_.shape[0], -1), O.repeat(R))[0]
    f = (mu * net._alpha[g]).reshape(6, P, n).mean(-1).cpu().numpy()   # contrasts of the model actually used
    f00, fa, fbA, fbB, fcis, ftr = f
    return [(int(pi[p]), int(pj[p]), fcis[p] - fa[p] - fbA[p] + f00[p], ftr[p] - fa[p] - fbB[p] + f00[p],
             fa[p] - f00[p], fbA[p] - f00[p]) for p in range(P)]

## 8. Split A — held-out families (main benchmark, ablations, deep ensemble, XAI contrasts)

In [ ]:
BASE_NAMES = ["Ridge (additive, clumped top-K)", "ElasticNet (PrediXcan-style)", "aFC-n-style haplotype model",
              "Pairwise SNPxSNP (phase-blind)", "Pairwise + cis", "Pairwise + trans", "Pairwise + cis + trans",
              "XGBoost (A/B-symmetrised)", "OriginSib stage 1+2 (EN + sparse cis/trans)"]
OOF = {m: np.full((G, nS), np.nan) for m in BASE_NAMES}
YTEST = np.full((G, nS), np.nan)
NET_PRED = defaultdict(dict)    # cfg -> seed -> (mu G x nS, var G x nS)
FOLD_CACHE = []
CF_ROWS = []
STAT_PRED = {}
SPARSE_ROWS = []
TRAIN_LOG = []
SEL_RECORD = defaultdict(list)
BASE_RESVAR = np.full((G, nS), np.nan)
LR_CHOSEN = None
t_split = time.time()
for k in range(CFG["N_FOLDS"]):
    tk = time.time()
    te = np.where(FOLDS == k)[0]; tr = np.where(FOLDS != k)[0]
    Rtr, Rte = prep_fold(tr, te)
    res = run_gene_models(tr, te, Rtr, Rte, seed=42 + k)
    t_stats = time.time() - tk
    log(f"  fold {k+1}: per-gene statistical models done in {t_stats/60:.1f} min")
    YTEST[:, te] = Rte.T
    for g, r in res.items():
        for m, p in r["preds"].items(): OOF[m][g, te] = p
        BASE_RESVAR[g, te] = r["base_res_var"]
        SEL_RECORD[g].append(VAR.vid.values[GENE_VARS[g][r["sel"]]].tolist())
    T = assemble_tensors(res)
    Yall = np.zeros((G, nS), np.float32); Yall[:, tr] = Rtr.T; Yall[:, te] = Rte.T
    BASES = build_bases(res, tr, te); base = BASES["cis+trans"]
    OOF["aFC-n-style haplotype model"][:, te] = fit_afcn(T["HA"], T["HB"], T["M"], Rtr.T, tr, te, FAM[tr])
    FT = FoldTensors(T, base, Yall, ORIENT)
    # inner validation families (early stopping) taken from TRAIN only
    trf = np.array(sorted(set(FAM[tr]))); rng_v = np.random.default_rng(100 + k)
    vf = set(rng_v.choice(trf, max(1, int(len(trf) * CFG["VAL_FRAC"])), replace=False))
    val = tr[np.isin(FAM[tr], list(vf))]; fit = tr[~np.isin(FAM[tr], list(vf))]
    if LR_CHOSEN is None:
        lr_scores = {}
        for lr in CFG["LR_GRID"]:
            net = train_compnet(FT, ABLATIONS[FULL], np.arange(G), fit, val, seed=42, lr=lr)
            lr_scores[lr] = net._best_val
        LR_CHOSEN = min(lr_scores, key=lr_scores.get)
        log(f"LR grid on fold-0 inner validation (train data only): {lr_scores} -> {LR_CHOSEN}")
    for name, cfg in ABLATIONS.items():
        if not cfg.get("nn", True):               # statistical stages only (deterministic)
            STAT_PRED.setdefault(name, np.full((G, nS), np.nan))[:, te] = BASES[cfg["int"]][:, te]
            continue
        FT.base = torch.tensor(BASES[cfg["int"]], dtype=torch.float32, device=DEVICE)
        seeds = CFG["SEEDS_FULL"] if name == FULL else CFG["SEEDS_ABLATION"]
        for sd in seeds:
            ts = time.time()
            net = train_compnet(FT, cfg, np.arange(G), fit, val, seed=sd, lr=LR_CHOSEN)
            mu, v = predict_compnet(net, FT, np.arange(G), te)
            if sd not in NET_PRED[name]:
                NET_PRED[name][sd] = (np.full((G, nS), np.nan), np.full((G, nS), np.nan))
            NET_PRED[name][sd][0][:, te] = mu; NET_PRED[name][sd][1][:, te] = v
            TRAIN_LOG.append(dict(fold=k, config=name, seed=sd, epochs=net._epochs, best_val=net._best_val,
                                  frac_genes_gated_on=net._alpha_frac,
                                  seconds=time.time() - ts))
            if name == FULL:
                for g in range(G):
                    vids = VAR.vid.values[GENE_VARS[g][res[g]["sel"]]]
                    ci = res[g]["int"]["cis+trans"]; kk = ci["k"]
                    if ci["coef"] is not None and kk >= 2:
                        P_ = len(pair_idx(kk)[0]); gam = ci["coef"][kk:kk + P_]; the = ci["coef"][kk + P_:kk + 2 * P_]
                        pmap = {(int(a), int(b)): q for q, (a, b) in enumerate(zip(*pair_idx(kk)))}
                    for a, b, ec, et, ea, eb in counterfactual_contrasts(net, FT, g, te, CFG["K_PAIR"], seed=sd):
                        lg = lt = 0.0
                        if ci["coef"] is not None and (a, b) in pmap:
                            # explicit stage-2 contributions (cis: both alt on one copy -> gamma; trans -> theta)
                            lg, lt = float(gam[pmap[(a, b)]]), float(the[pmap[(a, b)]])
                        CF_ROWS.append(dict(gene=GENES.index[g], fold=k, seed=sd, v1=vids[a], v2=vids[b],
                                            eps_cis=ec + lg, eps_trans=et + lt, nn_eps_cis=ec, nn_eps_trans=et,
                                            lasso_gamma=lg, lasso_theta=lt, main1=ea, main2=eb))
            del net
    FOLD_CACHE.append(dict(tr=tr, te=te, fit=fit, val=val, T=T, base=base, Yall=Yall))
    for g, r in res.items():
        ci = r["int"]["cis+trans"]
        if ci["coef"] is not None:
            vids = VAR.vid.values[GENE_VARS[g][r["sel"]]]; kk = ci["k"]; P_ = len(pair_idx(kk)[0])
            for q, (a, b) in enumerate(zip(*pair_idx(kk))):
                gm, th = ci["coef"][kk + q], ci["coef"][kk + P_ + q]
                if gm != 0 or th != 0:
                    SPARSE_ROWS.append(dict(gene=GENES.index[g], fold=k, v1=vids[a], v2=vids[b], gamma_cis=gm, theta_trans=th))
    del FT; gc.collect(); torch.cuda.empty_cache() if torch.cuda.is_available() else None
    log(f"Fold {k+1}/{CFG['N_FOLDS']} done in {(time.time()-tk)/60:.1f} min (statistics {t_stats/60:.1f} min, "
        f"neural {(time.time()-tk-t_stats)/60:.1f} min; n_train={len(tr)}, n_test={len(te)})")
log(f"Split A finished in {(time.time()-t_split)/60:.1f} min")
pd.DataFrame(TRAIN_LOG).to_csv(OUT / "tables" / "S3_training_log.csv", index=False)
SPARSE = pd.DataFrame(SPARSE_ROWS)
if len(SPARSE):   # selection stability of explicit cis/trans interaction coefficients across the 5 outer folds
    SPARSE = SPARSE.groupby(["gene", "v1", "v2"]).agg(n_folds_selected=("fold", "nunique"), gamma_cis=("gamma_cis", "mean"),
                                                       theta_trans=("theta_trans", "mean")).reset_index()
    SPARSE["gene_name"] = GENES.loc[SPARSE.gene, "gene_name"].values
    SPARSE.sort_values("n_folds_selected", ascending=False).to_csv(
        OUT / "candidate_interactions" / "stage2_sparse_interaction_coefficients.csv", index=False)
    log(f"Stage-2 sparse interactions: {len(SPARSE)} pairs selected in >=1 fold, "
        f"{int((SPARSE.n_folds_selected >= 3).sum())} in >=3/5 folds")

In [ ]:
# ---------------- metrics ----------------
def gene_metrics(Y, P, cols=None):
    rows = []
    for g in range(Y.shape[0]):
        y, p = Y[g], P[g]
        m = ~np.isnan(y) & ~np.isnan(p)
        if cols is not None: m &= cols
        y, p = y[m], p[m]; n = len(y)
        if n < 10:
            rows.append(dict(r=np.nan, rho=np.nan, rmse=np.nan, mae=np.nan, r2=np.nan, n=n, p_r=np.nan)); continue
        r = np.corrcoef(y, p)[0, 1] if np.std(p) > 1e-10 else 0.0
        r = 0.0 if np.isnan(r) else r
        rho = spearmanr(y, p).correlation if np.std(p) > 1e-10 else 0.0
        t = r * np.sqrt((n - 2) / max(1 - r ** 2, 1e-12))
        rows.append(dict(r=r, rho=0.0 if np.isnan(rho) else rho, rmse=np.sqrt(np.mean((y - p) ** 2)),
                         mae=np.mean(np.abs(y - p)), r2=1 - np.sum((y - p) ** 2) / np.sum((y - y.mean()) ** 2), n=n,
                         p_r=stats.t.sf(t, n - 2)))
    return pd.DataFrame(rows, index=GENES.index[:Y.shape[0]])

def ensemble(name, seeds=None):
    d = NET_PRED[name]; seeds = seeds or list(d.keys())
    mus = np.stack([d[s][0] for s in seeds]); vs = np.stack([d[s][1] for s in seeds])
    return mus.mean(0), vs.mean(0) + mus.var(0)

def boot_ci(x, B=None, seed=0, blocks=None):
    B = B or CFG["BOOTSTRAP"]; rng = np.random.default_rng(seed); x = np.asarray(x)
    ok = ~np.isnan(x); x = x[ok]
    if len(x) < 3: return (np.nan, np.nan)
    if blocks is None:
        bs = [x[rng.integers(0, len(x), len(x))].mean() for _ in range(B)]
    else:   # genomic-block bootstrap: neighbouring genes share LD / cis windows
        blocks = np.asarray(blocks)[ok]; ub = np.unique(blocks); idx = [np.where(blocks == b)[0] for b in ub]
        bs = [x[np.concatenate([idx[j] for j in rng.integers(0, len(ub), len(ub))])].mean() for _ in range(B)]
    return tuple(np.percentile(bs, [2.5, 97.5]))

PRED_ALL = dict(OOF)
for name in ABLATIONS:
    PRED_ALL[f"CompNet {name}"] = ensemble(name)[0] if name in NET_PRED else STAT_PRED[name]
PRED_ALL["OriginSib-CompNet (A6, 3-seed ensemble)"] = ensemble(FULL, CFG["SEEDS_ABLATION"])[0]
METRICS = {m: gene_metrics(YTEST, P) for m, P in PRED_ALL.items()}
REF = "ElasticNet (PrediXcan-style)"      # strongest additive baseline = reference for all ΔR²
def summarise(models, metrics=METRICS, ref=REF, label=""):
    rows = []
    ref_r2 = metrics[ref].r2.values
    for m in models:
        d = metrics[m]
        q = multipletests(d.p_r.fillna(1).values, method="fdr_bh")[1]
        dr2 = d.r2.values - ref_r2
        lo, hi = boot_ci(dr2, blocks=GENES.block.values[:len(dr2)])
        try:
            wp = wilcoxon(d.r2.values, ref_r2, nan_policy="omit").pvalue if m != ref else np.nan
        except Exception:
            wp = np.nan
        rows.append({"model": m, "genes": int(d.r.notna().sum()),
                     "Pearson r (mean±SE)": f"{d.r.mean():.4f} ± {d.r.std()/np.sqrt(d.r.notna().sum()):.4f}",
                     "Spearman ρ": round(d.rho.mean(), 4), "RMSE": round(d.rmse.mean(), 4), "MAE": round(d.mae.mean(), 4),
                     "R² mean": round(d.r2.mean(), 4), "R² median": round(d.r2.median(), 4),
                     "genes BH-sig (q<0.05, r>0)": int(((q < 0.05) & (d.r > 0)).sum()),
                     "ΔR² vs EN": round(np.nanmean(dr2), 5), "ΔR² 95% CI (block bootstrap)": f"[{lo:.5f}, {hi:.5f}]",
                     "Wilcoxon p vs EN": wp})
    return pd.DataFrame(rows)

T3 = summarise(BASE_NAMES + [f"CompNet {FULL}"])
T3["model"] = T3["model"].replace({f"CompNet {FULL}": "OriginSib-CompNet (A6, 5-seed deep ensemble)"})
# optional: precomputed sequence-model predictions (Enformer / Borzoi / AlphaGenome) attached as Kaggle input
SEQ_ROWS = []
if Path("/kaggle/input").exists():
    for f in Path("/kaggle/input").rglob("*"):
        if f.is_file() and re.search(r"(enformer|borzoi|alphagenome)", f.name, re.I) and f.suffix in (".csv", ".tsv", ".gz"):
            try:
                df = pd.read_csv(f, sep=None, engine="python")
                low = {c.lower(): c for c in df.columns}
                if not all(c in low for c in ["gene_id", "sample_id", "pred"]): continue
                df["g"] = df[low["gene_id"]].astype(str).str.replace(r"\.\d+$", "", regex=True)
                P = df.pivot_table(index="g", columns=low["sample_id"], values=low["pred"]).reindex(index=GENES.index, columns=S)
                d = gene_metrics(YTEST, P.values)
                SEQ_ROWS.append({"model": f"Frozen sequence model ({f.name})", "genes": int(P.notna().any(axis=1).sum()),
                                 "Pearson r (mean±SE)": f"{d.r.mean():.4f}", "Spearman ρ": round(d.rho.mean(), 4)})
            except Exception as e:
                log("seq-model file skipped:", f.name, e)
if not SEQ_ROWS:
    SEQ_ROWS = [{"model": "Frozen Enformer / Borzoi / AlphaGenome",
                 "Pearson r (mean±SE)": "not run: needs sequence-model inference; attach a CSV (gene_id,sample_id,pred) "
                                        "whose name contains enformer/borzoi/alphagenome to include it"}]
T3 = pd.concat([T3, pd.DataFrame(SEQ_ROWS)], ignore_index=True)
save_table(T3, "T3_main_expression_prediction_baselines")
print(T3.to_string())

abl_names = [f"CompNet {n}" for n in ABLATIONS] + ["OriginSib-CompNet (A6, 3-seed ensemble)"]
T5 = summarise(abl_names)
seed_sd = []
for name in [n for n in ABLATIONS if n in NET_PRED]:
    per_seed = [gene_metrics(YTEST, NET_PRED[name][s][0]).r2.mean() for s in NET_PRED[name]]
    seed_sd.append(dict(model=f"CompNet {name}", n_seeds=len(per_seed), **{"R² per-seed mean": np.mean(per_seed),
                                                                         "R² seed SD": np.std(per_seed)}))
T5 = T5.merge(pd.DataFrame(seed_sd), on="model", how="left")
save_table(T5, "T5_ablations_A0_A8")
print(T5[["model", "R² mean", "ΔR² vs EN", "ΔR² 95% CI (block bootstrap)", "Wilcoxon p vs EN", "R² seed SD"]].to_string())

# per-population performance under Split A (E8, part 1)
pop_rows = []
for p in POP_LEVELS:
    cols = POP == p
    for m in ["Ridge (additive, clumped top-K)", REF, f"CompNet {FULL}"]:
        d = gene_metrics(YTEST, PRED_ALL[m], cols)
        pop_rows.append(dict(split="A (held-out families)", population=p, n=int(cols.sum()), model=m.replace(f"CompNet {FULL}", "OriginSib-CompNet"),
                             r_mean=d.r.mean(), r2_mean=d.r2.mean()))
POP_A = pd.DataFrame(pop_rows)

## 9. Split B (held-out genes / genomic blocks) and Split C (cross-ancestry, leave-one-population-out)

In [ ]:
# ---- Split B: network never sees the test genes (contiguous 2 Mb blocks), nor the test individuals ----
SPLIT_B = None
if CFG["RUN_SPLIT_B"]:
    ub = np.array(sorted(GENES.block.unique())); rngb = np.random.default_rng(7); rngb.shuffle(ub)
    BFOLD = {b: i % CFG["N_FOLDS"] for i, b in enumerate(ub)}
    gfold = np.array([BFOLD[b] for b in GENES.block.values])
    PB = np.full((G, nS), np.nan); PB_seen = np.full((G, nS), np.nan); PB_ridge = np.full((G, nS), np.nan)
    A6_seen = ensemble(FULL, CFG["SEEDS_SPLIT_BC"])[0]
    for k, fc in enumerate(FOLD_CACHE):
        test_genes = np.where(gfold == k)[0]; train_genes = np.where(gfold != k)[0]
        if len(test_genes) == 0 or len(train_genes) == 0: continue
        FT = FoldTensors(fc["T"], fc["base"], fc["Yall"], ORIENT)
        mus = []
        for sd in CFG["SEEDS_SPLIT_BC"]:
            net = train_compnet(FT, ABLATIONS[FULL], train_genes, fc["fit"], fc["val"], seed=sd, lr=LR_CHOSEN)
            mus.append(predict_compnet(net, FT, test_genes, fc["te"])[0]); del net
        PB[np.ix_(test_genes, fc["te"])] = np.mean(mus, 0)
        PB_seen[np.ix_(test_genes, fc["te"])] = A6_seen[np.ix_(test_genes, fc["te"])]
        PB_ridge[np.ix_(test_genes, fc["te"])] = OOF[REF][np.ix_(test_genes, fc["te"])]
        del FT; gc.collect()
    mB = {"ElasticNet (additive base)": gene_metrics(YTEST, PB_ridge), "CompNet, gene seen in training (Split A)": gene_metrics(YTEST, PB_seen),
          "CompNet, gene held out (Split B)": gene_metrics(YTEST, PB)}
    rows = []
    for m, d in mB.items():
        dr2 = d.r2.values - mB["ElasticNet (additive base)"].r2.values
        lo, hi = boot_ci(dr2, blocks=GENES.block.values)
        rows.append(dict(split="B (held-out genes x held-out families)", model=m, r_mean=d.r.mean(), r2_mean=d.r2.mean(),
                         dR2_vs_EN=np.nanmean(dr2), dR2_CI=f"[{lo:.5f}, {hi:.5f}]"))
    SPLIT_B = pd.DataFrame(rows); print(SPLIT_B.to_string())
    log("Split B finished")

# ---- Split C: leave one population out (train on the rest, no population indicator covariates) ----
SPLIT_C = None
if CFG["RUN_SPLIT_C"] and len(POP_LEVELS) > 1:
    rows = []
    for p in POP_LEVELS:
        te = np.where(POP == p)[0]; tr = np.where(POP != p)[0]
        if len(te) < 20: continue
        Rtr, Rte = prep_fold(tr, te, use_pop=False)
        res = run_gene_models(tr, te, Rtr, Rte, seed=11, models=(), heads=("cis+trans",))
        T = assemble_tensors(res)
        Yall = np.zeros((G, nS), np.float32); Yall[:, tr] = Rtr.T; Yall[:, te] = Rte.T
        base = build_bases(res, tr, te)["cis+trans"]
        FT = FoldTensors(T, base, Yall, ORIENT)
        trf = np.array(sorted(set(FAM[tr]))); rv = np.random.default_rng(5)
        vf = set(rv.choice(trf, max(1, int(len(trf) * CFG["VAL_FRAC"])), replace=False))
        val = tr[np.isin(FAM[tr], list(vf))]; fit = tr[~np.isin(FAM[tr], list(vf))]
        mus = []
        for sd in CFG["SEEDS_SPLIT_BC"]:
            net = train_compnet(FT, ABLATIONS[FULL], np.arange(G), fit, val, seed=sd, lr=LR_CHOSEN)
            mus.append(predict_compnet(net, FT, np.arange(G), te)[0]); del net
        Yt = np.full((G, nS), np.nan); Yt[:, te] = Rte.T
        Pr = np.full((G, nS), np.nan)
        for g, r in res.items(): Pr[g, te] = r["preds"]["ElasticNet (PrediXcan-style)"]
        Pn = np.full((G, nS), np.nan); Pn[:, te] = np.mean(mus, 0)
        dr, dn = gene_metrics(Yt, Pr), gene_metrics(Yt, Pn)
        dr2 = dn.r2.values - dr.r2.values; lo, hi = boot_ci(dr2, blocks=GENES.block.values)
        rows.append(dict(split="C (leave-one-population-out)", population=p, n=len(te),
                         EN_r=dr.r.mean(), EN_r2=dr.r2.mean(), compnet_r=dn.r.mean(), compnet_r2=dn.r2.mean(),
                         dR2=np.nanmean(dr2), dR2_CI=f"[{lo:.5f}, {hi:.5f}]"))
        del FT; gc.collect()
        log(f"Split C: held-out {p} done")
    SPLIT_C = pd.DataFrame(rows); print(SPLIT_C.to_string())

T6_parts = [POP_A]
if SPLIT_B is not None: T6_parts.append(SPLIT_B)
if SPLIT_C is not None: T6_parts.append(SPLIT_C)
T6 = save_table(pd.concat(T6_parts, ignore_index=True), "T6_population_and_gene_heldout")

## 10. E5 — Interaction / compensation scan with discovery → replication and negative controls

In [ ]:
def ols_hc3(X, y, contrasts):
    """OLS with HC3 robust SEs. Columns with ~zero variance (except the intercept) are dropped.
       contrasts: list of dict {col_index: weight}. Returns list of (est, se, p) or NaNs."""
    n = X.shape[0]
    keep = [0] + [j for j in range(1, X.shape[1]) if X[:, j].std() > 1e-9]
    Xk = X[:, keep]; pos = {c: i for i, c in enumerate(keep)}
    XtX_inv = np.linalg.pinv(Xk.T @ Xk)
    beta = XtX_inv @ Xk.T @ y; e = y - Xk @ beta
    h = np.clip(np.einsum("ij,jk,ik->i", Xk, XtX_inv, Xk), 0, 0.99)
    meat = (Xk * (e ** 2 / (1 - h) ** 2)[:, None]).T @ Xk
    V = XtX_inv @ meat @ XtX_inv
    df = max(n - Xk.shape[1], 1); out = []
    for c in contrasts:
        if any(j not in pos for j in c):
            out.append((np.nan, np.nan, np.nan)); continue
        w = np.zeros(len(keep))
        for j, v in c.items(): w[pos[j]] = v
        est = w @ beta; se = np.sqrt(max(w @ V @ w, 1e-300))
        out.append((est, se, 2 * stats.t.sf(abs(est / se), df)))
    return out, beta, pos

def resid_var(X, j):
    """variance of column j after projecting out the other columns (for power)."""
    o = [c for c in range(X.shape[1]) if c != j and (c == 0 or X[:, c].std() > 1e-9)]
    b = np.linalg.lstsq(X[:, o], X[:, j], rcond=None)[0]
    return float(np.var(X[:, j] - X[:, o] @ b))

def scan_gene(g, idx, y, mode="top", pairs_fixed=None, cond_fixed=None, shuffle_phase=False, seed=0):
    rng = np.random.default_rng(seed + 31 * g)
    Hg = H_GENE[g][idx]; hA = Hg[:, :, 0].astype(float); hB = Hg[:, :, 1].astype(float)
    if shuffle_phase:
        s = (rng.random(hA.shape) < 0.5) & (hA != hB); hA, hB = np.where(s, hB, hA), np.where(s, hA, hB)
    D = hA + hB; n = len(y)
    sd = D.std(0); ok = sd > 1e-6
    Zs = np.zeros_like(D); Zs[:, ok] = (D[:, ok] - D[:, ok].mean(0)) / sd[ok]
    r = Zs.T @ ((y - y.mean()) / (y.std() + 1e-12)) / n; r[~ok] = 0
    if pairs_fixed is None:
        cand = clump(Zs, r, ok, CFG["K_PAIR"], CFG["PAIR_MAX_R2"])
        if mode == "random": cand = random_matched(VMETA[g][:, 1], ok, cand, rng)
        cond = [c for c in clump(Zs, r, ok, CFG["N_COND_EQTL"] + CFG["K_PAIR"], CFG["CLUMP_R2"])][:CFG["N_COND_EQTL"] + 2]
        pairs = list(itertools.combinations(cand, 2))
    else:
        pairs, cond = pairs_fixed, cond_fixed
    rows = []
    for i, j in pairs:
        if not (ok[i] and ok[j]): continue
        if (Zs[:, i] @ Zs[:, j] / n) ** 2 >= CFG["PAIR_MAX_R2"]: continue
        ci, cj = D[:, i] > 0, D[:, j] > 0
        cells = [(ci & cj).sum(), (ci & ~cj).sum(), (~ci & cj).sum(), (~ci & ~cj).sum()]
        if min(cells) < CFG["MIN_CELL"]: continue
        cc = [c for c in cond if c not in (i, j)][:CFG["N_COND_EQTL"]]
        Cv = D[:, cc] if cc else np.zeros((n, 0))
        di, dj = D[:, i], D[:, j]; hi, hj = hA[:, i] * hB[:, i], hA[:, j] * hB[:, j]
        cis = hA[:, i] * hA[:, j] + hB[:, i] * hB[:, j]; trn = hA[:, i] * hB[:, j] + hB[:, i] * hA[:, j]
        one = np.ones((n, 1))
        X0 = np.column_stack([one, di, dj, Cv])
        (m1, m2), _, _ = ols_hc3(X0, y, [{1: 1}, {2: 1}])
        X1 = np.column_stack([one, di, dj, hi, hj, di * dj, Cv])
        (t1,), _, _ = ols_hc3(X1, y, [{5: 1}])
        dh = (di == 1) & (dj == 1); n_coup = int((dh & (cis == 1)).sum()); n_rep = int((dh & (trn == 1)).sum())
        t2 = bc = bt = (np.nan, np.nan, np.nan)
        if n_coup >= CFG["MIN_PHASE_CELL"] and n_rep >= CFG["MIN_PHASE_CELL"]:
            X2 = np.column_stack([one, di, dj, hi, hj, cis, trn, Cv])
            (t2, bc, bt), _, _ = ols_hc3(X2, y, [{5: 1, 6: -1}, {5: 1}, {6: 1}])
        v = resid_var(X1, 5)
        cm = [y[(di == a) & (dj == b)].mean() if ((di == a) & (dj == b)).sum() >= 3 else np.nan
              for a, b in [(0, 0), (1, 0), (0, 1), (1, 1)]]
        rows.append(dict(gi=g, gene=GENES.index[g], i=i, j=j, v1=VAR.vid.values[GENE_VARS[g][i]], v2=VAR.vid.values[GENE_VARS[g][j]],
                         r2_ld=float((Zs[:, i] @ Zs[:, j] / n) ** 2), n=n, n11=cells[0], n10=cells[1], n01=cells[2], n00=cells[3],
                         n_coupling=n_coup, n_repulsion=n_rep, beta1=m1[0], beta2=m2[0], p_beta1=m1[2], p_beta2=m2[2],
                         gamma=t1[0], gamma_se=t1[1], p_int=t1[2], cis_minus_trans=t2[0], cmt_se=t2[1], p_cmt=t2[2],
                         beta_cis=bc[0], beta_trans=bt[0], resid_var_int=v,
                         y00=cm[0], y10=cm[1], y01=cm[2], y11=cm[3], cond=tuple(cc), pair=(i, j)))
    return rows, (pairs, cond)

def run_scan(idx, Y, mode="top", fixed=None, shuffle_phase=False, seed=0):
    out = Parallel(n_jobs=N_JOBS, backend="loky", batch_size=8)(
        delayed(scan_gene)(g, idx, Y[:, g], mode, *(fixed[g] if fixed else (None, None)), shuffle_phase, seed)
        for g in range(G))
    rows = [r for o in out for r in o[0]]
    return pd.DataFrame(rows), {g: o[1] for g, o in enumerate(out)}

def lambda_gc(p):
    p = np.asarray(p); p = p[~np.isnan(p)]
    return float(np.median(chi2.isf(p, 1)) / chi2.ppf(0.5, 1)) if len(p) else np.nan

# discovery / replication halves: whole families, stratified by population
HALF = make_folds(FAM, POP, 2, seed=99)
disc, rep = np.where(HALF == 0)[0], np.where(HALF == 1)[0]
Rd, _ = prep_fold(disc, np.array([], int)); Rr, _ = prep_fold(rep, np.array([], int))
SCAN, FIX = run_scan(disc, Rd, "top")
log(f"Discovery scan: {len(SCAN)} testable pairs in {SCAN.gene.nunique() if len(SCAN) else 0} genes "
    f"(n_disc={len(disc)}, n_rep={len(rep)})")
if len(SCAN):
    SCAN["q_int"] = multipletests(SCAN.p_int.fillna(1), method="fdr_bh")[1]
    m = SCAN.p_cmt.notna()
    SCAN["q_cmt"] = np.nan
    if m.any(): SCAN.loc[m, "q_cmt"] = multipletests(SCAN.loc[m, "p_cmt"], method="fdr_bh")[1]
    # replication: same pairs and conditioning variants, independent families
    fixed = {g: ([tuple(p) for p in SCAN[SCAN.gi == g].pair], list(FIX[g][1])) for g in range(G)}
    REP, _ = run_scan(rep, Rr, fixed=fixed)
    REP = REP.set_index(["gene", "v1", "v2"])
    for c in ["gamma", "gamma_se", "p_int", "cis_minus_trans", "cmt_se", "p_cmt", "n"]:
        SCAN["rep_" + c] = [REP[c].get((a, b, cc), np.nan) if (a, b, cc) in REP.index else np.nan
                            for a, b, cc in zip(SCAN.gene, SCAN.v1, SCAN.v2)]
    def one_sided(est, se, ref):
        z = est / se; return np.where(np.sign(est) == np.sign(ref), norm.sf(np.abs(z)), 1 - norm.sf(np.abs(z)))
    SCAN["rep_p1_int"] = one_sided(SCAN.rep_gamma, SCAN.rep_gamma_se, SCAN.gamma)
    SCAN["rep_p1_cmt"] = one_sided(SCAN.rep_cis_minus_trans, SCAN.rep_cmt_se, SCAN.cis_minus_trans)
    w1, w2 = 1 / SCAN.gamma_se ** 2, 1 / SCAN.rep_gamma_se ** 2
    SCAN["meta_gamma"] = (SCAN.gamma * w1 + SCAN.rep_gamma * w2) / (w1 + w2); SCAN["meta_se"] = 1 / np.sqrt(w1 + w2)
    SCAN["meta_p"] = 2 * norm.sf(np.abs(SCAN.meta_gamma / SCAN.meta_se))
    SCAN["Q_het_p"] = chi2.sf((SCAN.gamma - SCAN.meta_gamma) ** 2 * w1 + (SCAN.rep_gamma - SCAN.meta_gamma) ** 2 * w2, 1)
    # pattern classification (screening heuristics, NOT proof of compensation)
    b1, b2, gm = SCAN.beta1, SCAN.beta2, SCAN.gamma
    joint = b1 + b2 + gm
    SCAN["opposing_main_effects"] = (b1 * b2 < 0)
    SCAN["restorative"] = (b1 * b2 < 0) & (np.abs(joint) < np.maximum(np.abs(b1), np.abs(b2)))
    SCAN["buffering"] = (b1 * b2 > 0) & (np.sign(gm) != np.sign(b1 + b2)) & (np.abs(joint) < np.abs(b1 + b2))
    SCAN["synergistic"] = (np.sign(gm) == np.sign(b1 + b2)) & (np.abs(joint) > np.abs(b1 + b2))
    SCAN["pattern"] = np.select([SCAN.restorative, SCAN.buffering, SCAN.synergistic],
                                ["restorative (opposing effects, joint near reference)", "buffering (sub-additive)",
                                 "synergistic (super-additive)"], "other")
    # cCRE support
    vid_ccre = dict(zip(VAR.vid, VAR.ccre))
    SCAN["both_in_ccre"] = [bool(vid_ccre.get(a, False) and vid_ccre.get(b, False)) for a, b in zip(SCAN.v1, SCAN.v2)]
    SCAN["any_in_ccre"] = [bool(vid_ccre.get(a, False) or vid_ccre.get(b, False)) for a, b in zip(SCAN.v1, SCAN.v2)]
    # power: minimum detectable interaction (SD units) at 80% power, Bonferroni over tests
    a_b = 0.05 / max(len(SCAN), 1)
    SCAN["MDE80_bonf"] = (norm.isf(a_b / 2) + norm.isf(0.2)) / np.sqrt(SCAN.n * SCAN.resid_var_int.clip(1e-9))

# ---------- negative controls ----------
NEG = []
if len(SCAN):
    NEG.append(dict(test="Real data: SNPxSNP interaction (T1)", n_tests=len(SCAN), lambda_gc=lambda_gc(SCAN.p_int),
                    n_q05=int((SCAN.q_int < CFG["FDR"]).sum()), frac_p05=float((SCAN.p_int < 0.05).mean())))
    pc = SCAN.p_cmt.dropna()
    NEG.append(dict(test="Real data: cis-vs-trans phase test (T2)", n_tests=len(pc), lambda_gc=lambda_gc(pc),
                    n_q05=int((SCAN.q_cmt < CFG["FDR"]).sum()), frac_p05=float((pc < 0.05).mean()) if len(pc) else np.nan))
    PERM_P = []
    fixed_disc = {g: ([tuple(p) for p in SCAN[SCAN.gi == g].pair], list(FIX[g][1])) for g in range(G)}
    for b in range(CFG["N_PERM"]):
        rngp = np.random.default_rng(1000 + b); Yp = Rd.copy()
        for p in POP_LEVELS:
            ii = np.where(POP[disc] == p)[0]; Yp[ii] = Yp[rngp.permutation(ii)]
        Pm, _ = run_scan(disc, Yp, fixed=fixed_disc)
        if len(Pm): PERM_P.append(Pm.p_int.values)
    if PERM_P:
        allp = np.concatenate(PERM_P)
        NEG.append(dict(test=f"Permuted expression within population (x{len(PERM_P)})", n_tests=len(allp),
                        lambda_gc=lambda_gc(allp), n_q05=np.mean([(multipletests(p, method='fdr_bh')[1] < CFG['FDR']).sum() for p in PERM_P]),
                        frac_p05=float((allp < 0.05).mean())))
        SCAN["p_int_empirical"] = [(1 + (allp <= p).sum()) / (1 + len(allp)) for p in SCAN.p_int]
    RND, _ = run_scan(disc, Rd, "random", seed=5)
    if len(RND):
        NEG.append(dict(test="Random MAF-matched variant pairs", n_tests=len(RND), lambda_gc=lambda_gc(RND.p_int),
                        n_q05=int((multipletests(RND.p_int.fillna(1), method="fdr_bh")[1] < CFG["FDR"]).sum()),
                        frac_p05=float((RND.p_int < 0.05).mean())))
    SHF, _ = run_scan(disc, Rd, fixed=fixed_disc, shuffle_phase=True, seed=9)
    if len(SHF):
        ps = SHF.p_cmt.dropna()
        NEG.append(dict(test="Shuffled phase: cis-vs-trans test", n_tests=len(ps), lambda_gc=lambda_gc(ps),
                        n_q05=int((multipletests(ps, method='fdr_bh')[1] < CFG["FDR"]).sum()) if len(ps) else 0,
                        frac_p05=float((ps < 0.05).mean()) if len(ps) else np.nan))
NEG = pd.DataFrame(NEG)
save_table(NEG, "S2_statistical_calibration_negative_controls"); print(NEG.to_string())

In [ ]:
# ---------- cross-population consistency of the top pairs ----------
CROSS = pd.DataFrame()
if len(SCAN):
    top = SCAN.sort_values("p_int").head(40)
    sup_levels = [s for s in sorted(set(SUPER)) if (SUPER == s).sum() >= 60]
    res_by = {}
    for sp in sup_levels:
        ii = np.where(SUPER == sp)[0]
        Rs, _ = prep_fold(ii, np.array([], int), use_pop=True)
        fx = {g: ([tuple(p) for p in top[top.gi == g].pair], list(FIX[g][1])) for g in range(G)}
        R, _ = run_scan(ii, Rs, fixed=fx)
        if len(R): res_by[sp] = R.set_index(["gene", "v1", "v2"])
    rows = []
    for _, r in top.iterrows():
        d = dict(gene=r.gene, v1=r.v1, v2=r.v2, disc_gamma=r.gamma)
        ests = []
        for sp, R in res_by.items():
            key = (r.gene, r.v1, r.v2)
            if key in R.index:
                d[f"gamma_{sp}"] = R.loc[key, "gamma"]; d[f"se_{sp}"] = R.loc[key, "gamma_se"]
                ests.append((R.loc[key, "gamma"], R.loc[key, "gamma_se"]))
        if len(ests) >= 2:
            e = np.array(ests); w = 1 / e[:, 1] ** 2; mbar = (e[:, 0] * w).sum() / w.sum()
            d["Q_p_cross_pop"] = chi2.sf(((e[:, 0] - mbar) ** 2 * w).sum(), len(e) - 1)
            d["same_sign_all"] = bool(np.all(np.sign(e[:, 0]) == np.sign(r.gamma)))
        rows.append(d)
    CROSS = pd.DataFrame(rows)
    save_table(CROSS, "S2b_cross_population_interaction_consistency")

# ---------- model-implied (XAI) vs statistical interactions ----------
CF = pd.DataFrame(CF_ROWS)
XAI_AGREE = {}
if len(CF):
    CFm = CF.groupby(["gene", "v1", "v2"])[["eps_cis", "eps_trans", "main1", "main2"]].mean().reset_index()
    CFm["eps_mean"] = (CFm.eps_cis + CFm.eps_trans) / 2; CFm["eps_cis_minus_trans"] = CFm.eps_cis - CFm.eps_trans
    CFm.to_csv(OUT / "candidate_interactions" / "model_counterfactual_contrasts.csv", index=False)
    if len(SCAN):
        J = SCAN.merge(CFm, on=["gene", "v1", "v2"], how="inner")
        if len(J) >= 10:
            XAI_AGREE = dict(n_pairs=len(J), spearman_gamma_vs_eps=spearmanr(J.gamma, J.eps_mean).correlation,
                             spearman_cmt=spearmanr(J.cis_minus_trans.fillna(0), J.eps_cis_minus_trans).correlation,
                             sign_agreement=float((np.sign(J.gamma) == np.sign(J.eps_mean)).mean()))
            SCAN = SCAN.merge(CFm[["gene", "v1", "v2", "eps_mean", "eps_cis_minus_trans"]], on=["gene", "v1", "v2"], how="left")
log("Model-vs-statistics interaction agreement:", XAI_AGREE)

# ---------- evidence levels (blueprint §16) ----------
if len(SCAN):
    L2 = SCAN.q_int < CFG["FDR"]
    L2c = SCAN.q_cmt.fillna(1) < CFG["FDR"]
    rep_int = L2 & (SCAN.rep_p1_int < 0.05)
    rep_cmt = L2c & (SCAN.rep_p1_cmt.fillna(1) < 0.05)
    SCAN["evidence_level"] = 1
    SCAN.loc[L2 | L2c, "evidence_level"] = 2
    SCAN.loc[rep_int | rep_cmt, "evidence_level"] = 3
    SCAN.loc[(rep_int | rep_cmt) & SCAN.both_in_ccre, "evidence_level"] = 4
    SCAN["evidence_label"] = SCAN.evidence_level.map({
        1: "L1 tested / predicted candidate only", 2: "L2 statistical interaction (discovery FDR)",
        3: "L3 replicated association (independent families)",
        4: "L4* replicated + ENCODE cCRE regulatory annotation (not perturbation)"})
    SCAN["interaction_type"] = np.where(L2c | rep_cmt, "phase-dependent (cis≠trans)", "diploid (phase-blind)")
    cols7 = ["gene", "v1", "v2", "r2_ld", "n11", "n_coupling", "n_repulsion", "beta1", "beta2", "gamma", "gamma_se",
             "p_int", "q_int", "rep_gamma", "rep_p1_int", "meta_gamma", "meta_p", "Q_het_p", "cis_minus_trans", "q_cmt",
             "rep_p1_cmt", "pattern", "both_in_ccre", "MDE80_bonf", "evidence_level", "evidence_label", "interaction_type"]
    cols7 += [c for c in ["eps_mean", "eps_cis_minus_trans", "p_int_empirical"] if c in SCAN.columns]
    T7 = SCAN.sort_values(["evidence_level", "p_int"], ascending=[False, True])[cols7].head(50)
    T7.insert(1, "gene_name", GENES.loc[T7.gene, "gene_name"].values)
    save_table(T7, "T7_candidate_interactions_with_evidence")
    SCAN.drop(columns=["pair", "cond"]).to_csv(OUT / "candidate_interactions" / "full_interaction_scan.csv.gz", index=False)
    print("Evidence level counts:", SCAN.evidence_level.value_counts().sort_index().to_dict())
    print(T7.head(15).to_string())
else:
    T7 = pd.DataFrame()

## 11. E6 parent-aware vs parent-blind, exploratory POE, E9 functional consistency, optional MPRA, E10 calibration

In [ ]:
# ---------- E6 / RQ5 ----------
E6 = {}
orig_ind = np.where(np.abs(ORIENT).sum(1) > 0)[0]
E6["n_individuals_with_verified_origin"] = len(orig_ind)
if len(orig_ind) >= 10:
    cols = np.zeros(nS, bool); cols[orig_ind] = True
    a4 = gene_metrics(YTEST, PRED_ALL["CompNet A4 cis+trans + context"], cols)
    a6 = gene_metrics(YTEST, PRED_ALL[f"CompNet {FULL}"], cols)
    E6.update(parent_blind_A4_r2=a4.r2.mean(), parent_aware_A6_r2=a6.r2.mean(), delta=a6.r2.mean() - a4.r2.mean())
    poe_rows = []
    for g in range(G):
        ii = orig_ind[ORIENT[orig_ind, g] != 0]
        if len(ii) < 20: continue
        Hg = H_GENE[g][ii]; o = ORIENT[ii, g]
        pat = np.where(o[:, None] > 0, Hg[:, :, 0], Hg[:, :, 1]).astype(float)
        mat = np.where(o[:, None] > 0, Hg[:, :, 1], Hg[:, :, 0]).astype(float)
        y = YTEST[g, ii]; ok = ~np.isnan(y)
        lead = int(np.argmax(np.abs([np.corrcoef(pat[ok, j] + mat[ok, j], y[ok])[0, 1] if (pat[ok, j] + mat[ok, j]).std() > 0 else 0
                                     for j in range(pat.shape[1])])))
        X = np.column_stack([np.ones(ok.sum()), pat[ok, lead], mat[ok, lead]])
        (c,), _, _ = ols_hc3(X, y[ok], [{1: 1, 2: -1}])
        poe_rows.append(dict(gene=GENES.index[g], n=int(ok.sum()), lead=VAR.vid.values[GENE_VARS[g][lead]],
                             pat_minus_mat=c[0], se=c[1], p=c[2]))
    POE = pd.DataFrame(poe_rows)
    if len(POE):
        POE["q"] = multipletests(POE.p.fillna(1), method="fdr_bh")[1]; save_table(POE, "S4_exploratory_parent_of_origin")
        E6.update(n_genes_poe_tested=len(POE), n_poe_q05=int((POE.q < 0.05).sum()))
else:
    E6["status"] = ("RQ5 not testable: fewer than 10 expression-cohort individuals with Mendelian-verified parental "
                    "origin. Parent-of-origin claims are therefore NOT made; the diploid analyses (RQ1-4, RQ6) stand alone.")
log("E6/RQ5:", E6)

# ---------- E9 functional consistency ----------
E9 = {}
if len(SCAN) and CCRE_OK:
    sig = SCAN.q_int < CFG["FDR"]; top = SCAN.p_int <= SCAN.p_int.quantile(0.05)
    for lab, m in [("FDR-significant", sig), ("top 5% p-value", top)]:
        a = int((m & SCAN.any_in_ccre).sum()); b = int((m & ~SCAN.any_in_ccre).sum())
        c = int((~m & SCAN.any_in_ccre).sum()); d = int((~m & ~SCAN.any_in_ccre).sum())
        if a + b > 0:
            orr, p = fisher_exact([[a, b], [c, d]])
            E9[lab] = dict(n=a + b, frac_ccre=a / (a + b), background=c / max(c + d, 1), OR=orr, p=p)
log("E9 cCRE enrichment:", E9)

# ---------- optional MPRA (Nature 2026 pairwise data or any 4-configuration table in /kaggle/input) ----------
MPRA_RES = {"status": "not available: attach a table with columns y00,y10,y01,y11 (optional se/chrom/pos1/pos2)"}
def find_mpra():
    if not Path("/kaggle/input").exists(): return None
    for f in Path("/kaggle/input").rglob("*"):
        if f.suffix.lower() in (".csv", ".tsv", ".txt", ".gz", ".xlsx") and f.is_file() and f.stat().st_size < 2e9:
            try:
                df = pd.read_excel(f) if f.suffix == ".xlsx" else pd.read_csv(f, sep=None, engine="python", nrows=5)
                low = {c.lower(): c for c in df.columns}
                if all(k in low for k in ["y00", "y10", "y01", "y11"]):
                    return pd.read_excel(f) if f.suffix == ".xlsx" else pd.read_csv(f, sep=None, engine="python"), f
            except Exception:
                continue
    return None
_m = find_mpra()
if _m is not None:
    df, f = _m; low = {c.lower(): c for c in df.columns}
    y00, y10, y01, y11 = [df[low[k]].astype(float).values for k in ["y00", "y10", "y01", "y11"]]
    eps = y11 - y10 - y01 + y00
    a, b, jt = y10 - y00, y01 - y00, y11 - y00
    restor = (a * b < 0) & (np.abs(jt) < np.maximum(np.abs(a), np.abs(b)))
    MPRA_RES = dict(status=f"loaded {f.name}", n_pairs=len(eps), eps_mean=float(np.nanmean(eps)),
                    eps_sd=float(np.nanstd(eps)), frac_restorative_screen=float(np.nanmean(restor)))
    if "se" in low:
        z = eps / df[low["se"]].astype(float).values; p = 2 * norm.sf(np.abs(z))
        MPRA_RES["n_q05"] = int((multipletests(np.nan_to_num(p, nan=1), method="fdr_bh")[1] < 0.05).sum())
    if all(k in low for k in ["chrom", "pos1", "pos2"]) and len(SCAN):
        key = {(str(c).replace("chr", ""), int(p1), int(p2)): e for c, p1, p2, e in
               zip(df[low["chrom"]], df[low["pos1"]], df[low["pos2"]], eps)}
        mt = []
        for _, r in SCAN.iterrows():
            c1, p1 = r.v1.split(":")[:2]; p2 = int(r.v2.split(":")[1])
            for kk in [(c1.replace("chr", ""), int(p1), p2), (c1.replace("chr", ""), p2, int(p1))]:
                if kk in key: mt.append((r.gamma, key[kk]))
        if len(mt) >= 5:
            mt = np.array(mt); MPRA_RES.update(n_matched=len(mt), sign_agreement=float((np.sign(mt[:, 0]) == np.sign(mt[:, 1])).mean()),
                                               spearman=spearmanr(mt[:, 0], mt[:, 1]).correlation)
T4 = save_table(pd.DataFrame([MPRA_RES]), "T4_MPRA_interaction_benchmark")
log("MPRA:", MPRA_RES)

# ---------- E10 calibration ----------
mu6, v6 = ensemble(FULL)
mask = ~np.isnan(YTEST) & ~np.isnan(mu6)
y_, m_, s_ = YTEST[mask], mu6[mask], np.sqrt(v6[mask])
rv = BASE_RESVAR[mask]; pr = OOF[REF][mask]
levels = [0.5, 0.8, 0.9, 0.95]
CAL = []
for lv in levels:
    zc = norm.ppf((1 + lv) / 2)
    CAL.append(dict(nominal=lv, CompNet_coverage=float((np.abs(y_ - m_) <= zc * s_).mean()),
                    CompNet_width=float(2 * zc * s_.mean()),
                    EN_homoscedastic_coverage=float((np.abs(y_ - pr) <= zc * np.sqrt(rv)).mean()),
                    EN_width=float(2 * zc * np.sqrt(rv).mean())))
CAL = pd.DataFrame(CAL)
NLL = dict(CompNet=float(np.mean(0.5 * np.log(2 * np.pi * s_ ** 2) + (y_ - m_) ** 2 / (2 * s_ ** 2))),
           EN=float(np.mean(0.5 * np.log(2 * np.pi * rv) + (y_ - pr) ** 2 / (2 * rv))))
CAL["NLL_CompNet"] = NLL["CompNet"]; CAL["NLL_EN"] = NLL["EN"]
save_table(CAL, "S5_calibration_E10"); print(CAL.to_string())

## 12. Tables 1–2 and the figures (all from computed results)

In [ ]:
T1 = pd.DataFrame([
    dict(item="1000G 30x pedigree samples", value=len(PED)),
    dict(item="VCF samples in file / kept", value=f"{len(VCF_SAMPLES)} kept"),
    dict(item="Complete trio records", value=len(TRIOS)),
    dict(item="Trios used for phasing/origin QC", value=len(ORIGIN_QC)),
    dict(item="Median informative sites per trio", value=ORIGIN_QC.n_informative.median() if len(ORIGIN_QC) else np.nan),
    dict(item="Median Mendelian error rate", value=ORIGIN_QC.mendel_error_rate.median() if len(ORIGIN_QC) else np.nan),
    dict(item="Median switch rate (trio-informative sites)", value=ORIGIN_QC.switch_rate.median() if len(ORIGIN_QC) else np.nan),
    dict(item="GEUVADIS expression samples", value=GEU.shape[1]),
    dict(item="Analysis cohort (GEUVADIS ∩ 1000G 30x)", value=nS),
    dict(item="Families (independent units)", value=len(set(FAM))),
    dict(item="Populations", value=", ".join(f"{p}:{(POP==p).sum()}" for p in POP_LEVELS)),
    dict(item="Individuals with verified parental origin", value=N_ORIGIN_IND),
    dict(item="Chromosomes", value=", ".join(CFG["CHROMS"])),
    dict(item="Genes analysed", value=G),
    dict(item=f"cis SNVs (MAF>={CFG['MAF_MIN']}, ±{W//1000} kb)", value=len(VAR)),
    dict(item="Median cis SNVs per gene", value=float(GENES.n_cis_variants.median())),
    dict(item="Annotation", value=ANNOT_SRC),
    dict(item="ENCODE cCREs", value="yes" if CCRE_OK else "not available"),
    dict(item="MAGE", value=MAGE_NOTE),
    dict(item="Interaction pairs tested (discovery)", value=len(SCAN)),
    dict(item="SELF_TEST (synthetic)", value=CFG["SELF_TEST"]),
])
save_table(T1, "T1_dataset_summary")
T2 = pd.DataFrame([
    dict(work="Haplotype-aware cis-regulatory modelling (Nat Commun 2024)", did="multi-eQTL haplotype effects (aFC-n)",
         this_work="aFC-n-style baseline; adds explicit cis vs trans-haplotype separation + phase-shuffle control"),
    dict(work="MAGE (Nature 2024)", did="multi-ancestry RNA-seq, eQTL/sQTL", this_work="external independent-person validation (if reachable)"),
    dict(work="Parent-of-origin effects (Nature 2025)", did="large-scale POE in biobanks",
         this_work="POE head only for Mendelian-verified origin; reported as exploratory"),
    dict(work="Functional dissection MPRA (Nature 2026)", did="non-additive effects of 2,522 nearby variant pairs (same construct)",
         this_work="tests whether natural diploid combinations show replicable non-additivity; MPRA as cis ground truth (optional)"),
    dict(work="Personal genome models (Genome Biol 2026)", did="limited generalisation to unseen genes",
         this_work="gene-held-out (Split B) + family-held-out (Split A) + cross-ancestry (Split C)"),
])
save_table(T2, "T2_literature_novelty_comparison")

# ---------------- Figure 1: workflow ----------------
fig, ax = plt.subplots(figsize=(11, 4.2)); ax.axis("off")
boxes = [(0.01, 0.55, "1000G 30× trios\n+ phased VCF", PAL[0]), (0.01, 0.1, "GEUVADIS RNA-seq\n(+ MAGE)", PAL[0]),
         (0.22, 0.55, "Module 1\nMendelian QC, origin", PAL[1]), (0.22, 0.1, "Leakage-safe INT\n+ covariates (train)", PAL[1]),
         (0.43, 0.32, "Baselines: Ridge, EN,\naFC-n, pairwise, XGB", PAL[2]), (0.62, 0.55, "OriginSib-CompNet\nadd+cis+trans+POE", PAL[3]),
         (0.62, 0.1, "Interaction scan\ndisc→rep, controls", PAL[4]), (0.82, 0.32, "Evidence levels\nL1–L4, figures", PAL[5])]
for x, y, t, c in boxes:
    ax.add_patch(FancyBboxPatch((x, y), 0.16, 0.3, boxstyle="round,pad=0.01", fc=c, ec="none", alpha=0.9))
    ax.text(x + 0.08, y + 0.15, t, ha="center", va="center", color="white", fontsize=9, weight="bold")
for (x1, y1), (x2, y2) in [((0.17, 0.7), (0.22, 0.7)), ((0.17, 0.25), (0.22, 0.25)), ((0.38, 0.7), (0.43, 0.5)),
                           ((0.38, 0.25), (0.43, 0.42)), ((0.59, 0.5), (0.62, 0.65)), ((0.59, 0.42), (0.62, 0.27)),
                           ((0.78, 0.7), (0.82, 0.5)), ((0.78, 0.25), (0.82, 0.42))]:
    ax.annotate("", (x2, y2), (x1, y1), arrowprops=dict(arrowstyle="->", lw=1.4, color="#444"))
ax.set_title("Figure 1 | OriginSib-Comp workflow", loc="left")
savefig(fig, "Fig1_workflow")

# ---------------- Figure 4: expression validation ----------------
fig, axs = plt.subplots(1, 3, figsize=(14, 4))
order = [m for m in BASE_NAMES] + [f"CompNet {FULL}"]
lab = [m.replace(f"CompNet {FULL}", "OriginSib-CompNet").replace(" (additive, clumped top-K)", "")
       .replace(" (PrediXcan-style)", "") for m in order]
axs[0].boxplot([METRICS[m].r2.clip(-0.2).dropna().values for m in order], vert=False, showfliers=False)
axs[0].set_yticks(range(1, len(lab) + 1)); axs[0].set_yticklabels(lab, fontsize=7)
axs[0].axvline(0, c="#999", lw=0.8); axs[0].set_xlabel("held-out R² per gene (Split A)"); axs[0].set_title("a  Model comparison")
d0, d1 = METRICS[REF].r2.values, METRICS[f"CompNet {FULL}"].r2.values
axs[1].scatter(d0, d1, s=8, c=PAL[0], alpha=0.6); lim = [min(np.nanmin(d0), np.nanmin(d1)), max(np.nanmax(d0), np.nanmax(d1))]
axs[1].plot(lim, lim, c="#999", lw=0.8); axs[1].set_xlabel("ElasticNet (PrediXcan-style) R²"); axs[1].set_ylabel("OriginSib-CompNet R²")
axs[1].set_title("b  Per-gene paired comparison")
gbest = int(np.nanargmax(d1)); yy, pp = YTEST[gbest], PRED_ALL[f"CompNet {FULL}"][gbest]
axs[2].scatter(pp, yy, s=8, c=[PAL[POP_LEVELS.index(p) % len(PAL)] for p in POP], alpha=0.7)
axs[2].set_xlabel("predicted (held-out)"); axs[2].set_ylabel("observed residual expression")
axs[2].set_title(f"c  {GENES.gene_name.iloc[gbest]}  r={METRICS[f'CompNet {FULL}'].r.iloc[gbest]:.2f}")
savefig(fig, "Fig4_expression_validation")

# ---------------- Figure 7: ablations ----------------
fig, ax = plt.subplots(figsize=(8, 4))
vals, los, his = [], [], []
for n in ABLATIONS:
    dr2 = METRICS[f"CompNet {n}"].r2.values - METRICS[REF].r2.values
    vals.append(np.nanmean(dr2)); lo, hi = boot_ci(dr2, blocks=GENES.block.values); los.append(lo); his.append(hi)
y = np.arange(len(vals))
ax.barh(y, vals, color=[PAL[3] if "neg" not in n else PAL[6] for n in ABLATIONS], alpha=0.85)
ax.errorbar(vals, y, xerr=[np.array(vals) - np.array(los), np.array(his) - np.array(vals)], fmt="none", c="k", lw=1)
ax.set_yticks(y); ax.set_yticklabels(list(ABLATIONS)); ax.axvline(0, c="#999", lw=0.8); ax.invert_yaxis()
ax.set_xlabel("ΔR² vs additive ElasticNet (mean over genes, 95% block-bootstrap CI)"); ax.set_title("Figure 7 | Ablations A0–A8")
savefig(fig, "Fig7_ablations")

# ---------------- Figure 5: robustness ----------------
fig, axs = plt.subplots(1, 2, figsize=(12, 4))
if SPLIT_B is not None:
    axs[0].bar(range(len(SPLIT_B)), SPLIT_B.r2_mean, color=PAL[:len(SPLIT_B)])
    axs[0].set_xticks(range(len(SPLIT_B))); axs[0].set_xticklabels([s.replace(", ", ",\n") for s in SPLIT_B.model], fontsize=7)
    axs[0].set_ylabel("mean held-out R²"); axs[0].set_title("a  Gene-held-out (Split B)")
if SPLIT_C is not None and len(SPLIT_C):
    x = np.arange(len(SPLIT_C)); axs[1].bar(x - 0.2, SPLIT_C.EN_r2, 0.4, label="ElasticNet (additive)", color=PAL[0])
    axs[1].bar(x + 0.2, SPLIT_C.compnet_r2, 0.4, label="OriginSib-CompNet", color=PAL[3])
    axs[1].set_xticks(x); axs[1].set_xticklabels(SPLIT_C.population); axs[1].legend(frameon=False)
    axs[1].set_ylabel("mean R² in held-out population"); axs[1].set_title("b  Leave-one-population-out (Split C)")
savefig(fig, "Fig5_robustness")

# ---------------- Figures 2,3,6 and supplementary (scan) ----------------
if len(SCAN):
    best = SCAN.sort_values(["evidence_level", "p_int"], ascending=[False, True]).iloc[0]
    g = int(best.gi); r = GENES.iloc[g]
    fig, ax = plt.subplots(figsize=(11, 3.4))
    pos = VAR.pos.values[GENE_VARS[g]]
    D = H_GENE[g][disc].sum(-1).astype(float); yv = Rd[:, g]
    rr = np.array([np.corrcoef(D[:, j], yv)[0, 1] if D[:, j].std() > 0 else 0 for j in range(D.shape[1])])
    pv = -np.log10(np.clip(2 * stats.t.sf(np.abs(rr * np.sqrt((len(yv) - 2) / (1 - rr ** 2 + 1e-12))), len(yv) - 2), 1e-300, 1))
    cc = VAR.ccre.values[GENE_VARS[g]]
    ax.scatter(pos[~cc] / 1e6, pv[~cc], s=9, c="#9aa", label="cis SNV")
    ax.scatter(pos[cc] / 1e6, pv[cc], s=12, c=PAL[4], label="SNV in ENCODE cCRE")
    for _, q in SCAN[SCAN.gi == g].nsmallest(6, "p_int").iterrows():
        x1, x2 = VAR.pos.values[GENE_VARS[g][q.i]] / 1e6, VAR.pos.values[GENE_VARS[g][q.j]] / 1e6
        h = max(pv) * 1.05; xs = np.linspace(x1, x2, 50)
        ax.plot(xs, h + 0.15 * h * np.sin(np.linspace(0, np.pi, 50)) * (1 if q.gamma > 0 else -1),
                c=PAL[1] if q.gamma > 0 else PAL[0], lw=1.2, alpha=0.8)
    ax.axvline(r.tss / 1e6, c="k", ls="--", lw=0.8); ax.text(r.tss / 1e6, 0, " TSS", fontsize=8)
    ax.set_xlabel(f"{r.chrom} position (Mb, GRCh38)"); ax.set_ylabel("-log10 p (marginal eQTL, discovery)")
    ax.set_title(f"Figure 2 | Genomic context of {r.gene_name}: top interacting pairs (arcs: red γ>0, blue γ<0)")
    ax.legend(frameon=False, fontsize=8); savefig(fig, "Fig2_genomic_context")

    topk = SCAN.sort_values(["evidence_level", "p_int"], ascending=[False, True]).head(6)
    fig, axs = plt.subplots(1, len(topk), figsize=(3 * len(topk), 3.2), squeeze=False)
    for ax, (_, q) in zip(axs[0], topk.iterrows()):
        obs = [q.y00, q.y10, q.y01, q.y11]; add = [q.y00, q.y10, q.y01, q.y10 + q.y01 - q.y00]
        ax.plot(range(4), obs, "o-", c=PAL[1], label="observed cell mean"); ax.plot(range(4), add, "s--", c="#888", label="additive expectation")
        ax.set_xticks(range(4)); ax.set_xticklabels(["00", "10", "01", "11"]); ax.set_title(f"{GENES.loc[q.gene,'gene_name']}\nγ={q.gamma:.2f}, q={q.q_int:.2g}", fontsize=8)
    axs[0][0].legend(frameon=False, fontsize=7); axs[0][0].set_ylabel("residual expression (SD)")
    fig.suptitle("Figure 3 | Additive expectation vs observed (single-copy genotype cells, discovery half)", x=0.01, ha="left", weight="bold")
    savefig(fig, "Fig3_interaction_effects")

    fp = SCAN.sort_values(["evidence_level", "p_int"], ascending=[False, True]).head(15)
    fig, ax = plt.subplots(figsize=(7, 0.35 * len(fp) + 1.5)); yy = np.arange(len(fp))
    ax.errorbar(fp.gamma, yy - 0.15, xerr=1.96 * fp.gamma_se, fmt="o", c=PAL[0], label="discovery")
    ax.errorbar(fp.rep_gamma, yy + 0.15, xerr=1.96 * fp.rep_gamma_se if "rep_gamma_se" in fp else 0, fmt="s", c=PAL[2], label="replication")
    ax.set_yticks(yy); ax.set_yticklabels([f"{GENES.loc[a,'gene_name']} {b.split(':')[1]}×{c.split(':')[1]} (L{l})"
                                           for a, b, c, l in zip(fp.gene, fp.v1, fp.v2, fp.evidence_level)], fontsize=7)
    ax.axvline(0, c="#999", lw=0.8); ax.invert_yaxis(); ax.legend(frameon=False); ax.set_xlabel("interaction γ (SD units, 95% CI)")
    ax.set_title("Figure 6 | Candidate interactions: discovery vs independent replication"); savefig(fig, "Fig6_candidates_forest")

    fig, axs = plt.subplots(1, 3, figsize=(14, 4))
    def qq(ax, p, lab, c):
        p = np.sort(p[~np.isnan(p)]); e = (np.arange(1, len(p) + 1) - 0.5) / len(p)
        ax.plot(-np.log10(e), -np.log10(p), ".", ms=3, c=c, label=f"{lab} (λ={lambda_gc(p):.2f})")
    qq(axs[0], SCAN.p_int.values, "real", PAL[1])
    if PERM_P: qq(axs[0], np.concatenate(PERM_P), "permuted", "#888")
    if len(RND): qq(axs[0], RND.p_int.values, "random pairs", PAL[0])
    qq(axs[1], SCAN.p_cmt.values, "cis−trans real", PAL[3])
    if len(SHF): qq(axs[1], SHF.p_cmt.values, "shuffled phase", "#888")
    for a in axs[:2]:
        m = a.get_xlim()[1]; a.plot([0, m], [0, m], c="k", lw=0.7); a.legend(frameon=False, fontsize=7)
        a.set_xlabel("expected -log10 p"); a.set_ylabel("observed -log10 p")
    axs[0].set_title("a  SNP×SNP interaction QQ"); axs[1].set_title("b  cis-vs-trans phase test QQ")
    bgrid = np.linspace(0.05, 0.8, 30); a_b = 0.05 / len(SCAN)
    for qv, c in [(0.25, PAL[0]), (0.5, PAL[2]), (0.75, PAL[1])]:
        nv = np.quantile(SCAN.n * SCAN.resid_var_int, qv)
        axs[2].plot(bgrid, ncx2.sf(chi2.isf(a_b, 1), 1, bgrid ** 2 * nv), c=c, label=f"{int(qv*100)}th pct pair")
    axs[2].axhline(0.8, c="#999", ls="--"); axs[2].set_xlabel("true interaction effect (SD)"); axs[2].set_ylabel("power (Bonferroni)")
    axs[2].legend(frameon=False, fontsize=7); axs[2].set_title("c  Interaction power analysis")
    savefig(fig, "FigS1_qq_controls_power")

    if XAI_AGREE:
        J = SCAN.dropna(subset=["eps_mean"])
        fig, axs = plt.subplots(1, 2, figsize=(10, 4))
        axs[0].scatter(J.gamma, J.eps_mean, s=8, alpha=0.6, c=PAL[3]); axs[0].axhline(0, c="#999"); axs[0].axvline(0, c="#999")
        axs[0].set_xlabel("statistical γ (discovery)"); axs[0].set_ylabel("model counterfactual ε (held-out)")
        axs[0].set_title(f"a  ρ={XAI_AGREE['spearman_gamma_vs_eps']:.2f}, sign agr.={XAI_AGREE['sign_agreement']:.2f}")
        axs[1].scatter(J.cis_minus_trans, J.eps_cis_minus_trans, s=8, alpha=0.6, c=PAL[4])
        axs[1].set_xlabel("statistical cis−trans"); axs[1].set_ylabel("model ε_cis − ε_trans"); axs[1].set_title("b  Phase-dependent component")
        fig.suptitle("Figure 9 | Explainable-AI interaction contrasts vs statistical estimates", x=0.01, ha="left", weight="bold")
        savefig(fig, "Fig9_xai_vs_statistics")
    if E9:
        fig, ax = plt.subplots(figsize=(5, 3.5))
        labs = list(E9); ax.bar(np.arange(len(labs)) - 0.2, [E9[l]["frac_ccre"] for l in labs], 0.4, label="candidates", color=PAL[1])
        ax.bar(np.arange(len(labs)) + 0.2, [E9[l]["background"] for l in labs], 0.4, label="background", color="#aaa")
        ax.set_xticks(range(len(labs))); ax.set_xticklabels([f"{l}\np={E9[l]['p']:.2g}" for l in labs])
        ax.set_ylabel("pairs with ≥1 SNV in cCRE"); ax.legend(frameon=False); ax.set_title("Figure 9b | ENCODE cCRE enrichment (E9)")
        savefig(fig, "Fig9b_ccre_enrichment")

# ---------------- Figure 10: calibration ----------------
fig, axs = plt.subplots(1, 2, figsize=(10, 4))
axs[0].plot(CAL.nominal, CAL.CompNet_coverage, "o-", c=PAL[3], label="OriginSib-CompNet (deep ensemble)")
axs[0].plot(CAL.nominal, CAL.EN_homoscedastic_coverage, "s-", c=PAL[0], label="ElasticNet (homoscedastic)")
axs[0].plot([0.4, 1], [0.4, 1], c="#999", ls="--"); axs[0].set_xlabel("nominal coverage"); axs[0].set_ylabel("empirical coverage")
axs[0].legend(frameon=False); axs[0].set_title("a  Interval calibration")
zz = (y_ - m_) / s_; axs[1].hist(zz.clip(-5, 5), 80, density=True, color=PAL[3], alpha=0.7)
xx = np.linspace(-5, 5, 200); axs[1].plot(xx, norm.pdf(xx), c="k"); axs[1].set_title("b  Standardised residuals (held-out)")
savefig(fig, "Fig10_calibration")

if len(ORIGIN_QC):
    fig, axs = plt.subplots(1, 3, figsize=(12, 3.2))
    axs[0].hist(ORIGIN_QC.frac_hap0_paternal, 30, color=PAL[0]); axs[0].set_xlabel("fraction hap0 = paternal")
    axs[1].hist(ORIGIN_QC.switch_rate, 30, color=PAL[1]); axs[1].set_xlabel("switch rate at informative sites")
    axs[2].hist(ORIGIN_QC.mendel_error_rate, 30, color=PAL[2]); axs[2].set_xlabel("Mendelian error rate")
    fig.suptitle("Supplementary | Module 1 trio phasing / parental-origin QC", x=0.01, ha="left", weight="bold")
    savefig(fig, "FigS2_origin_qc")
log("Figure 8 (allele-specific expression) is not produced: GEUVADIS allelic read counts are not downloaded by this "
    "notebook. ASE validation stays future work and is not claimed.")

## 13. External validation in MAGE (independent persons) — runs only if MAGE could be loaded

In [ ]:
EXT = {"status": MAGE_NOTE}
if MAGE is not None:
    try:
        ext_s = [s for s in MAGE.columns if s in VSIDX]
        indep = [s for s in ext_s if s not in set(S)]
        genes_ok = [g for g in range(G) if GENES.index[g] in MAGE.index]
        log(f"MAGE external: {len(ext_s)} genotyped, {len(indep)} not in GEUVADIS, {len(genes_ok)} shared genes")
        if len(indep) >= 50 and len(genes_ok) >= 20:
            all_idx = np.arange(nS)
            Rall, _ = prep_fold(all_idx, np.array([], int))
            res = run_gene_models(all_idx, np.array([], int), Rall, np.zeros((0, G)), seed=1, models=())
            Me = MAGE.loc[GENES.index[genes_ok], indep].T.values.astype(float)
            Me = np.where(np.isnan(Me), np.nanmedian(Me, 0, keepdims=True), Me)
            vi = np.array([VSIDX[s] for s in indep])
            Ze = np.column_stack([norm.ppf((rankdata(Me[:, j]) - 0.5) / len(indep)) for j in range(Me.shape[1])])
            gp = HAP[pc_var][:, vi, :].sum(-1).T.astype(float); gp = (gp - gp.mean(0)) / (gp.std(0) + 1e-9)
            Ug = np.linalg.svd(gp, full_matrices=False)[0][:, :CFG["N_GENO_PCS"]]
            Ue = np.linalg.svd((Ze - Ze.mean(0)) / (Ze.std(0) + 1e-9), full_matrices=False)[0][:, :CFG["N_EXPR_PCS"]]
            Xc = np.column_stack([np.ones(len(indep)), Ug, Ue]); Re = Ze - Xc @ np.linalg.lstsq(Xc, Ze, rcond=None)[0]
            rows = []
            for jj, g in enumerate(genes_ok):
                rr = res[g]
                if len(rr["sel"]) == 0: continue
                vidx = GENE_VARS[g][rr["sel"]]
                Dx = HAP[vidx][:, vi, :].sum(-1).T.astype(float)
                Dtr = H_GENE[g][:, rr["sel"], :].sum(-1).astype(float)
                pred = ridge_fp(Dtr, Rall[:, g], Dx)[0]
                rows.append(dict(gene=GENES.index[g], r_ridge=np.corrcoef(pred, Re[:, jj])[0, 1] if pred.std() > 0 else 0))
            EXTDF = pd.DataFrame(rows); save_table(EXTDF, "S6_MAGE_external_validation")
            EXT = dict(status="run", n_independent=len(indep), n_genes=len(EXTDF), mean_r_ridge=float(EXTDF.r_ridge.mean()),
                       frac_genes_r_gt_0=float((EXTDF.r_ridge > 0).mean()))
    except Exception as e:
        EXT = {"status": f"failed: {type(e).__name__}: {e}"}
log("External validation:", EXT)

## 14. Summary, reproducibility manifest, zip

In [ ]:
SUMMARY = dict(
    self_test=CFG["SELF_TEST"], runtime_min=round((time.time() - T0) / 60, 1), versions=VERSIONS, config=CFG,
    lr_chosen=LR_CHOSEN, n_individuals=nS, n_genes=G, n_variants=len(VAR), N1=N1, N2=N2, N3=N3,
    main=T3.set_index("model")[["R² mean", "ΔR² vs EN", "ΔR² 95% CI (block bootstrap)", "Wilcoxon p vs EN"]].dropna(how="all").to_dict("index"),
    ablations=T5.set_index("model")[["R² mean", "ΔR² vs EN", "ΔR² 95% CI (block bootstrap)"]].to_dict("index"),
    split_B=SPLIT_B.to_dict("records") if SPLIT_B is not None else None,
    split_C=SPLIT_C.to_dict("records") if SPLIT_C is not None else None,
    scan=dict(n_pairs=len(SCAN), evidence_counts=SCAN.evidence_level.value_counts().to_dict() if len(SCAN) else {},
              negative_controls=NEG.to_dict("records")),
    xai_agreement=XAI_AGREE, E6_poe=E6, E9=E9, MPRA=MPRA_RES, calibration=CAL.to_dict("records"), external=EXT,
)
def _js(o):
    if isinstance(o, (np.integer,)): return int(o)
    if isinstance(o, (np.floating,)): return None if np.isnan(o) else float(o)
    if isinstance(o, (np.bool_,)): return bool(o)
    return str(o)
with open(OUT / "results_summary.json", "w") as f: json.dump(SUMMARY, f, indent=2, default=_js)
pd.DataFrame(MANIFEST).to_csv(OUT / "data_manifest.csv", index=False)
with open(OUT / "run_log.txt", "w") as f: f.write("\n".join(LOG))
with open(OUT / "README_results.md", "w") as f:
    f.write(f"# OriginSib-Comp results\n\nSELF_TEST={CFG['SELF_TEST']} (if True these numbers are synthetic and must not be reported).\n\n")
    f.write("## Main prediction (Split A, held-out families)\n\n" + md(T3) + "\n\n")
    f.write("## Ablations\n\n" + md(T5) + "\n\n")
    f.write("## Robustness\n\n" + md(T6) + "\n\n")
    f.write("## Negative controls\n\n" + md(NEG) + "\n\n")
    if len(T7): f.write("## Top candidate interactions\n\n" + md(T7.head(20)) + "\n\n")
    f.write("## Claims policy\n\nL1 = model-only/tested; L2 = discovery FDR; L3 = replicated in independent families; "
            "L4* = L3 + ENCODE cCRE annotation (not perturbation). No Level-5 (mechanistic) claims. "
            f"Parent-of-origin: {E6}.\n")
zp = shutil.make_archive(str(WORK / "originsib_results"), "zip", OUT)
log(f"DONE in {(time.time()-T0)/60:.1f} min. Results: {OUT}  |  zip: {zp}")
print(json.dumps({k: SUMMARY[k] for k in ["n_individuals", "n_genes", "n_variants", "lr_chosen", "xai_agreement", "external"]},
                 indent=1, default=_js))